# Protenix: Open-Source AlphaFold3-Level Structure Prediction

## CSV Batch Processor for High-Throughput Predictions

Easy-to-use batch processing interface for [Protenix](https://github.com/bytedance/protenix), ByteDance's open-source implementation delivering AlphaFold3-level accuracy for biomolecular structure prediction. Protenix supports proteins, DNA, RNA, ligands (CCD codes and SMILES), ions, and post-translational modifications.

### **Key Features:**
- **AlphaFold3-Level Accuracy**: Competitive performance across diverse biomolecular systems
- **Batch Processing**: CSV-based workflow for processing hundreds of predictions efficiently
- **Comprehensive Support**: Proteins, DNA, RNA, ligands (CCD codes and SMILES), ions
- **Post-Translational Modifications**: Built-in support for common PTMs
- **Multiple Models**: Base and mini model variants with optional ESM embeddings
- **MSA Integration**: Integrated MSA generation via ColabFold or pre-computed MSAs from MSA Colab
- **Full PAE Output**: Complete Predicted Aligned Error (PAE) matrices included in all predictions
- **Google Drive Integration**: Automatic upload of results to your Drive
- **Open Source**: Apache 2.0 license for academic and commercial use

### **Workflow:**
1. **Prepare CSV**: Create input file with sequences, modifications, and constraints
2. **Configure**: Set prediction parameters (model, seeds, MSA, precision)
3. **Run Batch**: Automated processing with progress tracking
4. **Download**: Results automatically uploaded to Google Drive

### **CSV Input Format:**
- **Required columns**: `jobname`, `seq1_name`, `seq1_type`, `seq1`
- **Optional columns**: `seq1_copies`, `seq1_mods`, `pocket_binder`, `pocket_contacts`, `covalent_bonds`
- **Supports**: Up to 10 sequences per job (seq1 through seq10)
- **Sequence types**: protein, dna, rna, ligand (CCD code or SMILES)

### **Supported Modifications:**
- **PTMs**: Phosphorylation (SEP, TPO, PTR), Methylation (MLY, M3L), Acetylation (ALY), and more
- **Ligands**: ATP, GTP, NAD, FAD, SAM, Heme, and 20+ common molecules
- **Ions**: Ca2+, Mg2+, Zn2+, Fe2+/3+, and other metal ions
- **Glycans**: NAG, MAN, GAL, FUC, and other carbohydrate modifications
- **DNA/RNA Mods**: Methylation, oxidation, and other nucleotide modifications
- **Custom**: Upload your own reference CSV with additional modifications

### **Repository:**
- [Protenix GitHub Repository](https://github.com/bytedance/protenix)
- [Protenix PyPI Package](https://pypi.org/project/protenix/)
- [Protenix Paper](https://www.biorxiv.org/content/10.1101/2024.11.14.623560)

### **Citations:**

**Protenix:**

[Zhang Z, Liu J, Zhao H, et al. Protenix - Advancing Structure Prediction Through Open-Source Implementation of AlphaFold3. *bioRxiv*, 2024](https://www.biorxiv.org/content/10.1101/2024.11.14.623560)

**If using automatic MSA generation:**

[Mirdita M, Schutze K, Moriwaki Y, et al. ColabFold: making protein folding accessible to all. *Nature Methods*, 2022](https://doi.org/10.1038/s41592-022-01488-1)

---

### **Known Limitations:**
- **Ligands (CCD codes)**: Protenix requires the `CCD_` prefix for CCD codes (e.g., `CCD_ATP`, not `ATP`). The notebook automatically adds this prefix for known CCD codes. Bare CCD codes without the prefix are treated as SMILES and will fail with `SMILES Parse Error`. **Workaround**: For CCD codes not in the built-in reference table, use direct SMILES strings in your CSV. Jobs without ligands (protein-only, protein-DNA, protein-RNA, protein-ion) are unaffected.

### **Quick Start Guide:**

1. **Run Cell 1**: Kernel restart (if needed — skips if already installed)
2. **Run Cell 2**: Upload your CSV file and connect Google Drive Protenix (takes ~3-5 minutes)
3. **Run Cell 3**: Install
4. **Run Cell 4**: Initialize CSV processor and process your CSV
5. **Run Cell 5**: Configure prediction parameters
6. **Run Cell 6**: Execute batch predictions (results auto-uploaded to Drive)

**Example CSV structure:**
```csv
jobname,seq1_name,seq1_type,seq1,seq2_name,seq2_type,seq2
my_protein,proteinA,protein,MSEQVENCE...,ligandATP,ligand,ATP
complex_with_ion,proteinB,protein,MKLLVV...,mg_ion,ligand,MG
```

For proteins with PTMs: `seq1_mods` column with format `CHAIN:POSITION:CCD_CODE` (e.g., `A:10:SEP;A:25:PTR`)

---

**Ready to start? Run Cell 1 below!**

In [ ]:
#@title Cell 1: Kernel Restart (if needed)
import subprocess
import sys
import os
import re

# Restart marker to handle Colab runtime state
restart_marker = "/content/.protenix_install_restart"
is_post_restart = os.path.exists(restart_marker)

def run_cmd(cmd, desc):
    """Execute command with output suppression unless error"""
    print(f"[{desc}]")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED: {result.stderr[:500]}")
        return False
    print("OK")
    return True

def get_cuda_version():
    """Detect CUDA version from nvidia-smi"""
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            match = re.search(r'CUDA Version: (\d+\.\d+)', result.stdout)
            if match:
                version = match.group(1)
                major = int(version.split('.')[0])
                minor = int(version.split('.')[1])
                return major, minor, version
    except Exception as e:
        print(f"Could not detect CUDA version: {e}")
    return None, None, None

if not is_post_restart:
    # ============================================================
    # PRE-RESTART: PREFLIGHT CHECKS + TRIGGER RESTART
    # ============================================================
    print("=" * 60)
    print("PREFLIGHT CHECKS")
    print("=" * 60)

    # Check GPU
    cuda_major, cuda_minor, cuda_version = get_cuda_version()
    if cuda_major is None:
        print("No CUDA detected - cannot proceed")
        sys.exit(1)

    print(f"CUDA Version: {cuda_version}")
    print(f"   Driver CUDA: {cuda_major}.{cuda_minor}")

    # Check GPU type
    result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          capture_output=True, text=True)
    if result.returncode == 0:
        gpu_name = result.stdout.strip()
        print(f"GPU: {gpu_name}")

    # Check pre-loaded NumPy version
    result = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                          capture_output=True, text=True)
    if result.returncode == 0:
        numpy_ver = result.stdout.strip()
        print(f"\nPre-loaded NumPy: {numpy_ver}")
        if numpy_ver.startswith('1.'):
            print("   Colab has NumPy 1.x - Protenix needs NumPy 2.x")
            print("   Will upgrade after restart")
        else:
            print("   NumPy 2.x already present - good")

    # Check pre-loaded PyTorch
    result = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True)
    if result.returncode == 0:
        torch_ver = result.stdout.strip()
        print(f"Pre-loaded PyTorch: {torch_ver}")

    print(f"\nInstall plan:")
    print(f"   1. Restart runtime to clear cached module state")
    print(f"   2. Install system dependencies (kalign, hmmer)")
    print(f"   3. Install protenix (pulls PyTorch 2.7.1 + numpy 2.x + all deps)")
    print(f"   4. Verify installation")

    print("\n" + "=" * 60)
    print("RESTARTING RUNTIME TO PREPARE FOR INSTALLATION")
    print("=" * 60)
    print("Runtime will restart in 2 seconds")
    print("After restart: Run this cell again to install")
    print("=" * 60)

    with open(restart_marker, "w") as f:
        f.write("restarted")

    import time
    time.sleep(2)
    os.kill(os.getpid(), 9)



In [ ]:
#@title Cell 2: Upload CSV and Connect Google Drive
#@markdown Upload your CSV file and connect Google Drive. Then click **Run All Below** for hands-free execution.
%%time

import subprocess
import sys
import os

# Quick-install pydrive2 (not in Colab baseline)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydrive2"], capture_output=True, text=True)

from google.colab import files

# Configuration options
setup_google_drive = True #@param {type:"boolean"}
#@markdown - Setup Google Drive for automatic result upload

gdrive_folder_name = "Protenix_Results" #@param {type:"string"}
#@markdown - Google Drive folder name for batch results

msa_folder = ""  #@param {type:"string"}
#@markdown - **Pre-computed MSAs** (from MSA Colab). Leave empty for online MSA.
#@markdown Enter the output folder name from MSA Colab (e.g., `MSA_Results`).
#@markdown The notebook will find the `paired/` subfolder automatically and
#@markdown download + unzip from Google Drive if needed.

print("=" * 60)
print("UPLOAD CSV AND CONNECT GOOGLE DRIVE")
print("=" * 60)

print("\nUpload your input CSV file with job specifications")
print("Required columns: jobname, seq1_name, seq1_type, seq1")
print("Optional: seq1_copies, seq1_mods, pocket_binder, pocket_contacts, covalent_bonds")
print("Supports up to 10 sequences per job (seq1 through seq10)")

uploaded_csv = files.upload()

if not uploaded_csv:
    raise ValueError("No CSV file uploaded")

csv_filename = list(uploaded_csv.keys())[0]
print(f"\nUploaded: {csv_filename}")

# Setup Google Drive
drive = None
if setup_google_drive:
    try:
        from pydrive2.drive import GoogleDrive
        from pydrive2.auth import GoogleAuth
        from google.colab import auth
        from oauth2client.client import GoogleCredentials

        print("\nSetting up Google Drive...")
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)
        print("Google Drive connected")
    except Exception as e:
        print(f"Google Drive setup failed: {e}")
        drive = None

# ============================================================
# RESOLVE PRE-COMPUTED MSA FOLDER
# ============================================================
import os as _os

def resolve_msa_folder(msa_folder_input, _drive=None):
    """Resolve user's msa_folder input to the actual paired/ directory path."""
    if not msa_folder_input or not msa_folder_input.strip():
        return ''

    name = msa_folder_input.strip().rstrip('/')

    if name.startswith('/'):
        base_path = name
    else:
        base_path = f"/content/{name}"

    if _os.path.isdir(base_path):
        paired = _os.path.join(base_path, 'paired')
        if _os.path.isdir(paired) and _os.listdir(paired):
            print(f"   MSA folder resolved: {paired}")
            return paired
        if _os.path.basename(base_path) == 'paired' and _os.listdir(base_path):
            print(f"   MSA folder resolved: {base_path}")
            return base_path

    zip_path = f"{base_path}.zip"
    if not _os.path.isfile(zip_path):
        zip_path = f"/content/{_os.path.basename(name)}.zip"

    if _os.path.isfile(zip_path):
        print(f"   Found local zip: {zip_path}")
        import zipfile as _zf
        extract_to = _os.path.dirname(zip_path) or '/content'
        with _zf.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_to)
        print(f"   Unzipped to {extract_to}")
        paired = _os.path.join(base_path, 'paired')
        if _os.path.isdir(paired) and _os.listdir(paired):
            print(f"   MSA folder resolved: {paired}")
            return paired

    if _drive:
        zip_filename = f"{_os.path.basename(name)}.zip"
        print(f"   Searching Google Drive for {zip_filename}...")
        try:
            file_list = _drive.ListFile({
                'q': f"title='{zip_filename}' and trashed=false"
            }).GetList()
            if file_list:
                gdrive_file = file_list[0]
                local_zip = f"/content/{zip_filename}"
                print(f"   Downloading {zip_filename} from Google Drive...")
                gdrive_file.GetContentFile(local_zip)
                print(f"   Downloaded ({_os.path.getsize(local_zip) / 1024 / 1024:.1f} MB)")

                import zipfile as _zf
                with _zf.ZipFile(local_zip, 'r') as zf:
                    zf.extractall('/content')
                print(f"   Unzipped to /content/")

                paired = _os.path.join(base_path, 'paired')
                if _os.path.isdir(paired) and _os.listdir(paired):
                    print(f"   MSA folder resolved: {paired}")
                    return paired
            else:
                print(f"   {zip_filename} not found on Google Drive")
        except Exception as e:
            print(f"   Google Drive download failed: {e}")

    print(f"   WARNING: Could not resolve MSA folder '{msa_folder_input}'")
    return ''

resolved_msa_folder = ''
if msa_folder:
    print(f"\nResolving MSA folder: {msa_folder}")
    resolved_msa_folder = resolve_msa_folder(msa_folder, drive)
    if resolved_msa_folder:
        print(f"   Using pre-computed MSAs from: {resolved_msa_folder}")
        msa_files = [f for f in _os.listdir(resolved_msa_folder) if f.endswith('_paired.a3m')]
        print(f"   Found {len(msa_files)} paired MSA files")

# Store settings (processor and jobs added in Cell 4)
global_settings = {
    'csv_filename': csv_filename,
    'drive': drive,
    'gdrive_folder_name': gdrive_folder_name,
    'msa_folder': resolved_msa_folder,
}

print("\n" + "=" * 60)
print(f"CSV uploaded, Google Drive {'connected' if drive else 'skipped'}.")
print("Run all cells below for hands-free install + predict.")
print("=" * 60)

In [ ]:
#@title Cell 3: Install Protenix
#@markdown Installs Protenix with pinned dependencies. Safe to re-run -- skips if already installed.
%%time

import subprocess
import sys
import os
import re
import time

def run_cmd(cmd, desc):
    """Execute command with output suppression unless error"""
    print(f"[{desc}]")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED: {result.stderr[:500]}")
        return False
    print("OK")
    return True

def get_cuda_version():
    """Detect CUDA version from nvidia-smi"""
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            match = re.search(r'CUDA Version: (\d+\.\d+)', result.stdout)
            if match:
                version = match.group(1)
                major = int(version.split('.')[0])
                minor = int(version.split('.')[1])
                return major, minor, version
    except Exception as e:
        print(f"Could not detect CUDA version: {e}")
    return None, None, None


restart_marker = "/content/.protenix_install_restart"
# ============================================================
# POST-RESTART: ACTUAL INSTALLATION
# ============================================================
print("=" * 60)
print("INSTALLING PROTENIX")
print("=" * 60)

# Detect CUDA version
cuda_major, cuda_minor, cuda_version = get_cuda_version()
if cuda_major is None:
    print("No CUDA detected - cannot proceed")
    sys.exit(1)

print(f"CUDA Version: {cuda_version}")

# Step 1: System dependencies
print("\n" + "=" * 60)
print("[1/5] System dependencies (kalign, hmmer)")
if not run_cmd(
    "apt-get update -qq && apt-get install -y -qq kalign hmmer",
    "Installing kalign and hmmer"
):
    print("System dependency installation failed")
    print("Trying individual packages...")
    run_cmd("apt-get install -y -qq kalign", "Installing kalign")
    run_cmd("apt-get install -y -qq hmmer", "Installing hmmer")

# Step 2a: Pre-install PyTorch from cu128 index (Blackwell sm_120 support)
# cu128 wheels ship kernels for sm_50 through sm_120, backward compatible with
# all Colab GPUs (T4, L4, A100, H100, G4). The default cu126 wheels only go up
# to sm_90 and fail on Blackwell with "no kernel image is available for execution".
print("\n" + "=" * 60)
print("[2/5] PyTorch 2.7.1 (cu128) + Protenix")
print("   Installing PyTorch from cu128 index (sm_120 Blackwell support)...")
if not run_cmd(
    f"{sys.executable} -m pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1"
    f" --extra-index-url https://download.pytorch.org/whl/cu128",
    "Installing PyTorch 2.7.1+cu128"
):
    print("PyTorch cu128 installation failed")
    sys.exit(1)

# Step 2b: Install protenix (torch already satisfied from cu128)
# protenix pulls: numpy>=2.0, cuEquivariance, DeepSpeed,
# fair-esm, rdkit, and many other dependencies
print("\n   Installing Protenix (with all remaining dependencies)...")
print("   This may take 3-5 minutes...")
if not run_cmd(
    f"{sys.executable} -m pip install protenix",
    "Installing protenix"
):
    print("Protenix installation failed")
    print("\nTrying with --no-cache-dir...")
    if not run_cmd(
        f"{sys.executable} -m pip install --no-cache-dir protenix",
        "Installing protenix (no cache)"
    ):
        print("Installation failed. Check error messages above.")
        sys.exit(1)

# ── Blackwell GPU (sm_120) compatibility patches ──
# On Blackwell GPUs:
#   - Layer norm CUDA extension needs sm_120 in arch targets for JIT compilation
#   - cuEquivariance's pynvml detection fails → patch fallback defaults
#   - triangle_multiplicative Triton kernels segfault → --trimul_kernel torch (in run cell)
#   - triangle_attention CUDA cubins work fine at full speed
try:
    _cap_r = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    _compute_cap = float(_cap_r.stdout.strip().splitlines()[0]) if _cap_r.returncode == 0 else 0
    if _compute_cap >= 12.0:
        print(chr(10) + '=' * 60)
        print('BLACKWELL GPU DETECTED (compute capability ' + str(_compute_cap) + ')')
        print('Applying Protenix Blackwell compatibility patches...')
        print('=' * 60)
        import site as _site, glob as _glob
        _sp = _site.getsitepackages()[0]

        # Patch 1: Layer norm CUDA arch targets — add sm_120 to _wanted list
        _ln_file = os.path.join(_sp, 'protenix/model/layer_norm/torch_ext_compile.py')
        if os.path.exists(_ln_file):
            with open(_ln_file) as _f:
                _src = _f.read()
            if '_wanted' in _src:
                # New-style (Protenix >= 1.0.9): dynamic arch detection with _wanted list
                if '("120"' not in _src:
                    if '("100", "100")]' in _src:
                        _src = _src.replace('("100", "100")]', '("100", "100"), ("120", "120")]')
                    else:
                        import re as _re
                        _src = _re.sub(r'(_wanted\s*=\s*\[.*?)\]', r'\1, ("120", "120")]', _src, flags=_re.DOTALL)
                # Ensure fallback ARCH_LIST includes 12.0
                if '"8.0;8.6;9.0;10.0;12.0"' not in _src:
                    import re as _re
                    _src = _re.sub(r'else\s+"[\d.;]+"', 'else "8.0;8.6;9.0;10.0;12.0"', _src)
                with open(_ln_file, 'w') as _f:
                    _f.write(_src)
                if '120' in open(_ln_file).read():
                    print(f'   {chr(0x2705)} Patch 1: Added sm_120 to layer_norm _wanted list')
                else:
                    print(f'   {chr(0x26a0)}  Patch 1: sm_120 may not have been added correctly')
            else:
                # Old-style (Protenix <= 1.0.4): static TORCH_CUDA_ARCH_LIST
                _src = _src.replace('"7.0;8.0"', '"7.0;8.0;8.6;9.0;10.0;12.0"')
                _NL = chr(10)
                _new_gc = _NL.join([
                    '            "-gencode",',
                    '            "arch=compute_70,code=sm_70",',
                    '            "-gencode",',
                    '            "arch=compute_80,code=sm_80",',
                    '            "-gencode",',
                    '            "arch=compute_86,code=sm_86",',
                    '            "-gencode",',
                    '            "arch=compute_90,code=sm_90",',
                    '            "-gencode",',
                    '            "arch=compute_100,code=sm_100",',
                    '            "-gencode",',
                    '            "arch=compute_120,code=sm_120",',
                ]) + _NL
                import re as _re
                _src = _re.sub(
                    r'("-gencode",\s*\n\s*"arch=compute_\d+,code=sm_\d+",?\s*\n?)+',
                    _new_gc, _src,
                )
                with open(_ln_file, 'w') as _f:
                    _f.write(_src)
                print(f'   {chr(0x2705)} Patch 1: Replaced static arch list + gencode flags (legacy path)')
        else:
            print(f'   {chr(0x26a0)}  {_ln_file} not found — Patch 1 skipped')

        # Patch 2: cuEquivariance GPU detection fallback defaults
        _cueq_files = _glob.glob(
            os.path.join(_sp, '**', 'cuequivariance_ops', 'triton', 'cache_manager.py'),
            recursive=True
        )
        if _cueq_files:
            _cueq_file = _cueq_files[0]
            with open(_cueq_file) as _f:
                _src = _f.read()
            _cueq_replacements = {
                '"NVIDIA RTX A6000"': '"Generic High-End GPU"',
                '"major": 8': '"major": 12',
                '"minor": 6': '"minor": 0',
                '"total_memory": 45': '"total_memory": 96',
                '"multi_processor_count": 84': '"multi_processor_count": 188',
                '"power_limit": 300': '"power_limit": 600',
                '"clock_rate": 2100': '"clock_rate": 3090',
            }
            _changed = False
            for _old, _new in _cueq_replacements.items():
                if _old in _src:
                    _src = _src.replace(_old, _new)
                    _changed = True
            if _changed:
                with open(_cueq_file, 'w') as _f:
                    _f.write(_src)
                print(f'   {chr(0x2705)} Patch 2: cuEquivariance fallback defaults updated')
            else:
                print(f'   {chr(0x2705)} Patch 2: cuEquivariance fallback already patched')
        else:
            print(f'   {chr(0x26a0)}  cuEquivariance not installed, skipping Patch 2')

        if 'global_settings' not in globals():
            global_settings = {}
        global_settings['is_blackwell'] = True
        print(chr(10) + '   Blackwell mixed-mode: triatt=cuEquivariance (full speed), trimul=torch (CLI flag in run cell)')
    else:
        if 'global_settings' not in globals():
            global_settings = {}
        global_settings['is_blackwell'] = False
except Exception as _e:
    print(f'   {chr(0x26a0)}  Blackwell patches skipped: {_e}')
    if 'global_settings' not in globals():
        global_settings = {}
    global_settings['is_blackwell'] = False


# Step 3: Install ipSAE_batch for interface analysis
print("\n" + "=" * 60)
print("[3/5] ipSAE_batch (interface scoring and visualization)")
# Pre-install vis deps that aren't in Protenix's tree
run_cmd(
    f"{sys.executable} -m pip install seaborn pycirclize python-igraph plotly",
    "Installing visualization dependencies"
)
if not run_cmd(
    f"{sys.executable} -m pip install git+https://github.com/JKourelis/ipSAE_batch.git",
    "Installing ipSAE_batch"
):
    print("WARNING: ipSAE_batch failed to install. Interface analysis will be unavailable.")

# Step 4: Verify NumPy compatibility
print("\n" + "=" * 60)
print("[4/5] Verifying NumPy compatibility")
result = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True
)
if result.returncode == 0:
    numpy_ver = result.stdout.strip()
    print(f"   NumPy version: {numpy_ver}")
    if numpy_ver.startswith('2.'):
        print(f"   NumPy 2.x confirmed - compatible with Protenix")
    else:
        print(f"   WARNING: NumPy {numpy_ver} detected")
        print(f"   Protenix requires NumPy 2.x. Attempting upgrade...")
        run_cmd(f"{sys.executable} -m pip install 'numpy>=2.0'", "Upgrading NumPy")
else:
    print("   WARNING: Could not check NumPy version")

# Step 5: Verify installation
print("\n" + "=" * 60)
print("[5/5] Verification")

# Test CLI
result = subprocess.run(
    ["protenix", "pred", "--help"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("   protenix pred --help: OK")
else:
    print(f"   protenix CLI check failed: {result.stderr[:200]}")
    # Try module invocation
    result2 = subprocess.run(
        [sys.executable, "-m", "protenix.predict", "--help"],
        capture_output=True, text=True
    )
    if result2.returncode == 0:
        print("   python -m protenix.predict --help: OK (using module invocation)")
    else:
        print("   WARNING: CLI verification failed. Protenix may still work.")


# Test ipSAE_batch
result = subprocess.run(["ipsae-batch", "--help"], capture_output=True, text=True)
if result.returncode == 0:
    print("   ipsae-batch CLI: OK")
else:
    print("   ipsae-batch CLI: NOT AVAILABLE (interface analysis will be skipped)")

# ============================================================
# ENVIRONMENT & VERSION LOG
# ============================================================
env_lines = []
env_lines.append("\n" + "=" * 60)
env_lines.append("ENVIRONMENT & INSTALLED VERSIONS")
env_lines.append("=" * 60)

# Python version
env_lines.append(f"\nPython: {sys.version.split()[0]}")

# GPU info
gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu_result.returncode == 0:
    env_lines.append(f"GPU: {gpu_result.stdout.strip()}")
env_lines.append(f"CUDA (driver max): {cuda_version}")

# PyTorch + CUDA runtime
result = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(f'PyTorch: {torch.__version__}'); "
     "print(f'CUDA available: {torch.cuda.is_available()}'); "
     "print(f'CUDA runtime: {torch.version.cuda}') if torch.cuda.is_available() else None"],
    capture_output=True, text=True
)
if result.returncode == 0:
    for line in result.stdout.strip().split('\n'):
        env_lines.append(f"   {line}")

# All relevant package versions via pip
result = subprocess.run(
    [sys.executable, "-m", "pip", "list", "--format=freeze"],
    capture_output=True, text=True
)
all_packages = result.stdout.strip().split('\n')

# Protenix + core deps
env_lines.append("\nProtenix stack:")
protenix_pkgs = [
    'protenix', 'numpy', 'torch', 'torchvision', 'torchaudio',
    'deepspeed', 'fair-esm', 'rdkit', 'scipy', 'pandas',
    'biopython', 'biotite', 'modelcif', 'gemmi', 'pdbeccdutils',
    'cuequivariance-ops-torch-cu12', 'cuequivariance-torch',
    'triton', 'optree', 'pydantic', 'wandb',
    'scikit-learn', 'scikit-learn-extra', 'ml-collections',
]
for pkg in protenix_pkgs:
    pkg_lower = pkg.lower().replace('-', '_')
    for line in all_packages:
        line_lower = line.lower().replace('-', '_')
        if line_lower.startswith(pkg_lower + '=='):
            env_lines.append(f"   {line}")
            break
    else:
        # Try partial match
        for line in all_packages:
            if pkg_lower in line.lower().replace('-', '_'):
                env_lines.append(f"   {line}")
                break

# ipSAE_batch + visualization deps
env_lines.append("\nipSAE_batch stack:")
ipsae_pkgs = [
    'ipsae-batch', 'seaborn', 'pycirclize', 'python-igraph', 'plotly',
    'matplotlib', 'networkx',
]
for pkg in ipsae_pkgs:
    pkg_lower = pkg.lower().replace('-', '_')
    for line in all_packages:
        line_lower = line.lower().replace('-', '_')
        if line_lower.startswith(pkg_lower + '=='):
            env_lines.append(f"   {line}")
            break
    else:
        for line in all_packages:
            if pkg_lower in line.lower().replace('-', '_'):
                env_lines.append(f"   {line}")
                break
        else:
            env_lines.append(f"   {pkg}: not installed")


# Note: full atom-level confidence output (PAE matrices, full_data_sample_N.json)
# is enabled via --need_atom_confidence True in the protenix pred CLI command (Cell 6).



# Cleanup
os.remove(restart_marker)

env_lines.append("\n" + "=" * 60)

# Write environment log to file and print
env_log = "\n".join(str(x) for x in env_lines)
print(env_log)
with open("/content/environment.txt", "w") as _ef:
    _ef.write(env_log + "\n")
print("INSTALLATION COMPLETE")
print("=" * 60)
print("Proceed to Cell 4.")


In [ ]:
#@title Cell 4: Protenix CSV Processor Setup
import pandas as pd
import os
import re
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from io import StringIO

class ProtenixJobProcessor:
    """Process CSV files for Protenix batch predictions.

    Converts unified CSV format to Protenix AF3-like JSON format.
    """

    EMBEDDED_REFERENCE = """Type,CCD Code,Name,Target Residue,Molecular Formula,All Atom Count,Heavy Atom Count,SMILES
PTM,SEP,Phosphoserine,SER,C3H8NO6P,18,11
PTM,TPO,Phosphothreonine,THR,C4H10NO6P,21,12
PTM,PTR,Phosphotyrosine,TYR,C9H12NO6P,28,17
PTM,MLY,N-Methyllysine,LYS,C7H16N2O2,27,11
PTM,ALY,N-Acetyllysine,LYS,C8H16N2O3,29,13
PTM,HYP,Hydroxyproline,PRO,C5H9NO3,18,9
PTM,M3L,N-Trimethyllysine,LYS,C9H20N2O2,33,13
PTM,MLZ,N-Methyllysine,LYS,C7H16N2O2,27,11
PTM,CSD,Cysteine sulfinic acid,CYS,C3H7NO4S,16,9
PTM,CSO,S-Hydroxycysteine,CYS,C3H7NO3S,15,8
PTM,CGU,Gamma-carboxyglutamic acid,GLU,C5H7NO6,21,12
PTM,FME,N-Formylmethionine,MET,C6H11NO3S,22,11
PTM,NEP,N-(phosphonoethyl)isoleucine,ILE,C8H18NO5P,32,15
PTM,HIC,4-Methyl-histidine,HIS,C7H11N3O2,23,12
PTM,CAS,S-(dimethylarsenic)cysteine,CYS,C5H11AsNO2S,20,9
PTM,HY3,3-Hydroxyproline,PRO,C5H9NO3,18,9
Ligand,ATP,Adenosine triphosphate,NA,C10H16N5O13P3,47,31,Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P@](O)(=O)O[P@@](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,ADP,Adenosine diphosphate,NA,C10H15N5O10P2,42,27,Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P@@](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,AMP,Adenosine monophosphate,NA,C10H14N5O7P,37,23,Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,GTP,Guanosine triphosphate,NA,C10H16N5O14P3,48,32,NC1=Nc2n(cnc2C(=O)N1)[C@@H]3O[C@H](CO[P](O)(=O)O[P](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,GDP,Guanosine diphosphate,NA,C10H15N5O11P2,43,28,NC1=Nc2n(cnc2C(=O)N1)[C@@H]3O[C@H](CO[P](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,GMP,Guanosine monophosphate,NA,C10H14N5O8P,38,24,NC1=Nc2n(cnc2C(=O)N1)[C@@H]3O[C@H](CO[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,CTP,Cytidine triphosphate,NA,C9H16N3O14P3,45,29,NC1=NC(=O)N(C=C1)[C@@H]2O[C@H](CO[P@@](O)(=O)O[P@@](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]2O
Ligand,CDP,Cytidine diphosphate,NA,C9H15N3O11P2,40,25,NC1=NC(=O)N(C=C1)[C@@H]2O[C@H](CO[P@](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]2O
Ligand,UTP,Uridine triphosphate,NA,C9H15N2O15P3,44,29,O[C@H]1[C@@H](O)[C@@H](O[C@@H]1CO[P@](O)(=O)O[P@](O)(=O)O[P](O)(O)=O)N2C=CC(=O)NC2=O
Ligand,UDP,Uridine diphosphate,NA,C9H14N2O12P2,39,25,O[C@H]1[C@@H](O)[C@@H](O[C@@H]1CO[P](O)(=O)O[P](O)(O)=O)N2C=CC(=O)NC2=O
Ligand,NAD,Nicotinamide adenine dinucleotide,NA,C21H27N7O14P2,71,44,NC(=O)c1ccc[n+](c1)[C@@H]2O[C@H](CO[P]([O-])(=O)O[P@](O)(=O)OC[C@H]3O[C@H]([C@H](O)[C@@H]3O)n4cnc5c(N)ncnc45)[C@@H](O)[C@H]2O
Ligand,NAP,NADP,NA,C21H28N7O17P3,86,55,NC(=O)c1ccc[n+](c1)[C@@H]2O[C@H](CO[P]([O-])(=O)O[P@@](O)(=O)OC[C@H]3O[C@H]([C@H](O[P](O)(O)=O)[C@@H]3O)n4cnc5c(N)ncnc45)[C@@H](O)[C@H]2O
Ligand,FAD,Flavin adenine dinucleotide,NA,C27H33N9O15P2,91,53,Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P@](O)(=O)O[P@@](O)(=O)OC[C@H]4O[C@H]([C@H](O)[C@@H]4O)n5cnc6c(N)ncnc56)c2cc1C
Ligand,FMN,Flavin mononucleotide,NA,C17H21N4O9P,52,31,Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P](O)(O)=O)c2cc1C
Ligand,HEM,Heme,NA,C34H32FeN4O4,75,43,CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O)C(=C8C=C)C)c(C)c7C=C
Ligand,SAH,S-Adenosyl-L-homocysteine,NA,C14H20N6O5S,46,26,N[C@@H](CCSC[C@H]1O[C@H]([C@H](O)[C@@H]1O)n2cnc3c(N)ncnc23)C(O)=O
Ligand,SAM,S-Adenosyl-L-methionine,NA,C15H22N6O5S,49,27,C[S@@+](CC[C@H](N)C([O-])=O)C[C@H]1O[C@H]([C@H](O)[C@@H]1O)n2cnc3c(N)ncnc23
Ligand,COA,Coenzyme A,NA,C21H36N7O16P3S,90,57,CC(C)(CO[P@@](O)(=O)O[P@](O)(=O)OC[C@H]1O[C@H]([C@H](O)[C@@H]1O[P](O)(O)=O)n2cnc3c(N)ncnc23)[C@@H](O)C(=O)NCCC(=O)NCCS
Ligand,ACO,Acetyl coenzyme A,NA,C23H38N7O17P3S,99,61,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)CO[P@@](O)(=O)O[P@](O)(=O)OC[C@H]1O[C@H]([C@H](O)[C@@H]1O[P](O)(O)=O)n2cnc3c(N)ncnc23
Ligand,PLP,Pyridoxal-5-phosphate,NA,C8H10NO6P,25,16,Cc1ncc(CO[P](O)(O)=O)c(C=O)c1O
Ligand,TPP,Thiamine diphosphate,NA,C12H19N4O7P2S,45,25,Cc1ncc(C[n+]2csc(CCO[P@@](O)(=O)O[P](O)(O)=O)c2C)c(N)n1
Ligand,BTN,Biotin,NA,C10H16N2O3S,32,16,OC(=O)CCCC[C@@H]1SC[C@@H]2NC(=O)N[C@H]12
Ligand,MTA,5-Methylthioadenosine,NA,C11H15N5O3S,35,20,CSC[C@H]1O[C@H]([C@H](O)[C@@H]1O)n2cnc3c(N)ncnc23
Ligand,THM,Thiamine,NA,C12H17ClN4OS,38,18,
Ion,MG,Magnesium ion,NA,Mg,1,1
Ion,ZN,Zinc ion,NA,Zn,1,1
Ion,CA,Calcium ion,NA,Ca,1,1
Ion,FE,Iron ion,NA,Fe,1,1
Ion,MN,Manganese ion,NA,Mn,1,1
Ion,CU,Copper ion,NA,Cu,1,1
Ion,CO,Cobalt ion,NA,Co,1,1
Ion,NI,Nickel ion,NA,Ni,1,1
Ion,K,Potassium ion,NA,K,1,1
Ion,NA,Sodium ion,NA,Na,1,1
Ion,CL,Chloride ion,NA,Cl,1,1
Glycan,NAG,N-Acetyl-D-glucosamine,NA,C8H15NO6,30,15
Glycan,MAN,D-Mannose,NA,C6H12O6,24,12
Glycan,FUC,L-Fucose,NA,C6H12O5,23,11
Glycan,GAL,D-Galactose,NA,C6H12O6,24,12
Glycan,SIA,N-Acetylneuraminic acid,NA,C11H19NO9,40,21
Glycan,GLC,D-Glucose,NA,C6H12O6,24,12
Glycan,BMA,beta-D-Mannose,NA,C6H12O6,24,12
Glycan,NDG,N-Acetyl-D-glucosamine,NA,C8H15NO6,30,15
Glycan,A2G,N-Acetyl-D-galactosamine,NA,C8H15NO6,30,15
Glycan,FUL,L-Fucose,NA,C6H12O5,23,11
DNA_Mod,5MC,5-Methylcytosine,DC,C10H15N3O7P,36,21
DNA_Mod,6MA,N6-Methyladenine,DA,C11H16N5O6P,39,23
DNA_Mod,5HMC,5-Hydroxymethylcytosine,DC,C10H15N3O8P,37,22
DNA_Mod,8OG,8-Oxoguanine,DG,C10H13N5O8P,37,24
DNA_Mod,M2G,N2-Methylguanine,DG,C11H16N5O7P,40,24
DNA_Mod,4MC,N4-Methylcytosine,DC,C10H15N3O7P,36,21
DNA_Mod,1MA,1-Methyladenine,DA,C11H16N5O6P,39,23
DNA_Mod,3MA,3-Methyladenine,DA,C11H16N5O6P,39,23
RNA_Mod,PSU,Pseudouridine,U,C9H12N2O9P,33,21
RNA_Mod,1MA,1-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,7MG,7-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,5MC,5-Methylcytidine,C,C10H15N3O8P,37,22
RNA_Mod,2MA,2-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,M2G,N2-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,M7G,7-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,M1A,1-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,OMC,2'-O-Methylcytidine,C,C10H15N3O8P,37,22
RNA_Mod,OMG,2'-O-Methylguanosine,G,C11H15N5O8P,40,25"""

    def __init__(self, reference_csv: Optional[str] = None):
        """Initialize processor with reference data"""
        if reference_csv:
            self.reference_data = pd.read_csv(StringIO(reference_csv))
        else:
            self.reference_data = pd.read_csv(StringIO(self.EMBEDDED_REFERENCE))

        self.ptm_lookup = self._create_lookup('PTM')
        self.ligand_lookup = self._create_lookup('Ligand')
        self.ion_lookup = self._create_lookup('Ion')
        self.glycan_lookup = self._create_lookup('Glycan')
        self.dna_mod_lookup = self._create_lookup('DNA_Mod')
        self.rna_mod_lookup = self._create_lookup('RNA_Mod')

        self.aa_3to1 = {
            'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
            'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
            'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
            'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'
        }

        self.amino_acids = set('ACDEFGHIKLMNPQRSTVWY')

    def _create_lookup(self, type_name: str) -> Dict[str, Dict[str, Any]]:
        """Create lookup dictionary for a specific type"""
        type_data = self.reference_data[self.reference_data['Type'] == type_name]
        lookup = {}
        for _, row in type_data.iterrows():
            if pd.notna(row['CCD Code']):
                entry = {
                    'name': row['Name'],
                    'target_residue': row['Target Residue'] if pd.notna(row['Target Residue']) else 'NA',
                    'heavy_atom_count': int(row['Heavy Atom Count']) if pd.notna(row['Heavy Atom Count']) else 0
                }
                if 'SMILES' in row and pd.notna(row.get('SMILES')) and str(row.get('SMILES', '')).strip():
                    entry['smiles'] = str(row['SMILES']).strip()
                lookup[row['CCD Code']] = entry
        return lookup

    def _validate_sequence_characters(self, sequence: str, seq_type: str) -> List[str]:
        """Validate sequence contains only allowed characters"""
        errors = []
        sequence = sequence.upper()

        if seq_type.lower() == 'protein':
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in self.amino_acids:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid amino acids: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        elif seq_type.lower() == 'dna':
            valid_bases = set('ATCG')
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in valid_bases:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid DNA bases: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        elif seq_type.lower() == 'rna':
            valid_bases = set('AUCG')
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in valid_bases:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid RNA bases: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        return errors

    def _is_smiles(self, ligand_string: str) -> bool:
        """Check if string is likely a SMILES representation"""
        if len(ligand_string) < 3:
            return False
        smiles_chars = set('[]()=#@+-\\/CNOPSFClBrI0123456789')
        return any(char in smiles_chars for char in ligand_string)

    def _remap_modification_chains(self, mod_string: str, name_mapping: Dict[str, str]) -> str:
        """Remap modification chain IDs from user names to A, B, C... format"""
        if pd.isna(mod_string) or str(mod_string).strip() == '':
            return mod_string
        mod_string = str(mod_string).strip()
        for old_name, new_chain_id in name_mapping.items():
            mod_string = mod_string.replace(f"{old_name}:", f"{new_chain_id}:")
        return mod_string

    def _remap_contact_chains(self, contacts_string: str, name_mapping: Dict[str, str]) -> str:
        """Remap contact chain IDs from user names to A, B, C... format"""
        if pd.isna(contacts_string) or str(contacts_string).strip() == '':
            return contacts_string
        contacts_string = str(contacts_string).strip()
        for old_name, new_chain_id in name_mapping.items():
            contacts_string = contacts_string.replace(f"{old_name}:", f"{new_chain_id}:")
        return contacts_string

    def _remap_bond_chains(self, bonds_string: str, name_mapping: Dict[str, str]) -> str:
        """Remap covalent bond chain IDs from user names to A, B, C... format"""
        if pd.isna(bonds_string) or str(bonds_string).strip() == '':
            return bonds_string
        bonds_string = str(bonds_string).strip()
        for old_name, new_chain_id in name_mapping.items():
            bonds_string = bonds_string.replace(f"{old_name}:", f"{new_chain_id}:")
        return bonds_string

    def _parse_modifications(self, mod_string: str, sequence: str, chain_id: str, seq_type: str) -> Tuple[List[Dict], List[str]]:
        """Parse modifications in format: CHAIN:POSITION:CCD_CODE"""
        errors = []
        mods = []

        if pd.isna(mod_string) or str(mod_string).strip() == '':
            return mods, errors

        mod_string = str(mod_string).strip()
        mod_entries = [e.strip() for e in mod_string.split(';') if e.strip()]

        if seq_type.lower() == 'protein':
            mod_lookups = [self.ptm_lookup, self.glycan_lookup]
            mod_types = ['PTM', 'Glycan']
        elif seq_type.lower() == 'dna':
            mod_lookups = [self.dna_mod_lookup]
            mod_types = ['DNA Mod']
        elif seq_type.lower() == 'rna':
            mod_lookups = [self.rna_mod_lookup]
            mod_types = ['RNA Mod']
        else:
            return mods, errors

        for entry in mod_entries:
            parts = entry.split(':')
            if len(parts) != 3:
                errors.append(f"Invalid modification format: '{entry}'. Use CHAIN:POSITION:CCD_CODE")
                continue

            mod_chain, pos_str, ccd_code = parts
            mod_chain = mod_chain.strip()
            ccd_code = ccd_code.strip()

            if mod_chain != chain_id:
                continue

            try:
                position = int(pos_str.strip())

                found = False
                for lookup, mod_type in zip(mod_lookups, mod_types):
                    if ccd_code in lookup:
                        found = True

                        if position < 1 or position > len(sequence):
                            errors.append(f"{mod_type} position {position} out of range (sequence length: {len(sequence)})")
                            continue

                        target_residue = lookup[ccd_code]['target_residue']
                        actual_residue = sequence[position - 1].upper()

                        if target_residue != 'NA':
                            if mod_type == 'Glycan':
                                if actual_residue not in ['N', 'T', 'S']:
                                    errors.append(f"Glycan {ccd_code} requires N/T/S but found {actual_residue} at position {position}")
                                    continue
                            elif mod_type == 'PTM':
                                target_1letter = self.aa_3to1.get(target_residue.upper())
                                if target_1letter and actual_residue != target_1letter:
                                    errors.append(f"PTM {ccd_code} targets {target_residue}({target_1letter}) but found {actual_residue} at position {position}")
                                    continue
                            else:
                                if len(target_residue) > 1 and target_residue.startswith('D'):
                                    target_residue = target_residue[1]
                                if actual_residue != target_residue:
                                    errors.append(f"{mod_type} {ccd_code} targets {target_residue} but found {actual_residue} at position {position}")
                                    continue

                        mods.append({
                            'chain_id': mod_chain,
                            'position': position,
                            'ccd': ccd_code
                        })
                        break

                if not found:
                    errors.append(f"Unknown modification code: '{ccd_code}'")

            except ValueError:
                errors.append(f"Invalid position in modification: '{entry}'")

        return mods, errors

    def _parse_pocket(self, binder: str, contacts_string: str) -> Tuple[Optional[Dict], List[str]]:
        """Parse pocket constraint"""
        errors = []

        if pd.isna(binder) or str(binder).strip() == '':
            return None, errors

        binder = str(binder).strip()

        if pd.isna(contacts_string) or str(contacts_string).strip() == '':
            errors.append(f"Pocket binder '{binder}' specified but no contacts provided")
            return None, errors

        contacts_string = str(contacts_string).strip()
        contact_entries = [e.strip() for e in contacts_string.split(';') if e.strip()]

        contacts = []
        for entry in contact_entries:
            parts = entry.split(':')
            if len(parts) != 2:
                errors.append(f"Invalid contact format: '{entry}'. Use CHAIN:RESIDUE")
                continue
            try:
                chain_id, residue_num = parts
                contacts.append([chain_id.strip(), int(residue_num.strip())])
            except ValueError:
                errors.append(f"Invalid contact specification: '{entry}'")

        if not contacts:
            errors.append(f"No valid contacts parsed for pocket binder '{binder}'")
            return None, errors

        return {
            'binder': binder,
            'contacts': contacts
        }, errors

    def _parse_covalent_bonds(self, bonds_string: str) -> Tuple[List[Dict], List[str]]:
        """Parse covalent bond constraints in format CHAIN:RES:ATOM:CHAIN:RES:ATOM"""
        errors = []
        bonds = []

        if pd.isna(bonds_string) or str(bonds_string).strip() == '':
            return bonds, errors

        bonds_string = str(bonds_string).strip()
        bond_entries = [e.strip() for e in bonds_string.split(';') if e.strip()]

        for entry in bond_entries:
            parts = entry.split(':')
            if len(parts) != 6:
                errors.append(f"Invalid covalent bond format: '{entry}'. Use CHAIN:RES:ATOM:CHAIN:RES:ATOM")
                continue
            try:
                chain1, res1, atom1, chain2, res2, atom2 = parts
                bonds.append({
                    'atom1': [chain1.strip(), int(res1.strip()), atom1.strip()],
                    'atom2': [chain2.strip(), int(res2.strip()), atom2.strip()]
                })
            except ValueError:
                errors.append(f"Invalid covalent bond specification: '{entry}'")

        return bonds, errors

    def _sanitize_jobname(self, jobname: str) -> Tuple[str, List[str]]:
        """Sanitize jobname for filesystem compatibility"""
        errors = []

        if pd.isna(jobname) or str(jobname).strip() == '':
            errors.append("Missing jobname")
            return '', errors

        original = str(jobname)
        sanitized = original.lower()
        sanitized = sanitized.replace('-', '_')
        sanitized = re.sub(r'[^a-z0-9_]', '', sanitized)

        if len(sanitized) > 128:
            sanitized = sanitized[:128]
            errors.append(f"Jobname truncated to 128 characters")

        if not sanitized.strip():
            errors.append(f"Jobname '{original}' became empty after sanitization")
            return '', errors

        return sanitized, errors

    def _generate_json(self, job: Dict) -> str:
        """Generate Protenix AF3-like JSON format.

        Protenix JSON format:
        [{
          "name": "jobname",
          "sequences": [
            {"proteinChain": {"sequence": "...", "count": N, "modifications": [...]}},
            {"dnaSequence": {"sequence": "...", "count": N}},
            {"rnaSequence": {"sequence": "...", "count": N}},
            {"ligand": {"ligand": "CCD_XXX", "count": N}},
            {"ion": {"ion": "MG", "count": N}}
          ],
          "covalent_bonds": [...]
        }]
        """
        sequences_json = []

        # Group sequences by type and identity for copy counting
        # Track which sequences we've processed (group identical ones)
        protein_groups = {}
        dna_groups = {}
        rna_groups = {}
        ligand_entries = []

        for seq in job['sequences']:
            seq_type = seq['type']

            if seq_type == 'protein':
                key = seq['sequence']
                if key not in protein_groups:
                    protein_groups[key] = {'count': 0, 'modifications': seq.get('modifications') or []}
                protein_groups[key]['count'] += 1

            elif seq_type == 'dna':
                key = seq['sequence']
                if key not in dna_groups:
                    dna_groups[key] = {'count': 0}
                dna_groups[key]['count'] += 1

            elif seq_type == 'rna':
                key = seq['sequence']
                if key not in rna_groups:
                    rna_groups[key] = {'count': 0}
                rna_groups[key]['count'] += 1

            elif seq_type == 'ligand':
                ligand_entries.append(seq)

        # Build protein chains
        for sequence, info in protein_groups.items():
            protein_entry = {
                "sequence": sequence,
                "count": info['count']
            }
            if info['modifications']:
                protein_entry["modifications"] = [
                    {"ptmType": f"CCD_{mod['ccd']}", "ptmPosition": mod['position']}
                    for mod in info['modifications']
                ]
            else:
                protein_entry["modifications"] = []
            sequences_json.append({"proteinChain": protein_entry})

        # Build DNA sequences
        for sequence, info in dna_groups.items():
            sequences_json.append({
                "dnaSequence": {
                    "sequence": sequence,
                    "count": info['count']
                }
            })

        # Build RNA sequences
        for sequence, info in rna_groups.items():
            sequences_json.append({
                "rnaSequence": {
                    "sequence": sequence,
                    "count": info['count']
                }
            })

        # Build ligands and ions
        # Group identical ligands for count
        ligand_groups = {}
        for seq in ligand_entries:
            if 'smiles' in seq:
                key = ('smiles', seq['smiles'])
            else:
                key = ('ccd', seq['ccd'])
            if key not in ligand_groups:
                ligand_groups[key] = 0
            ligand_groups[key] += 1

        for (lig_type, lig_value), count in ligand_groups.items():
            if lig_value in self.ion_lookup:
                sequences_json.append({
                    "ion": {
                        "ion": lig_value,
                        "count": count
                    }
                })
            elif any(c in lig_value for c in '()=#[]@/\\'):
                # Contains SMILES structural characters — pass as-is
                sequences_json.append({
                    "ligand": {
                        "ligand": lig_value,
                        "count": count
                    }
                })
            else:
                # CCD code — must have CCD_ prefix for Protenix
                sequences_json.append({
                    "ligand": {
                        "ligand": f"CCD_{lig_value}",
                        "count": count
                    }
                })

        # Build covalent bonds for Protenix format
        covalent_bonds_json = []
        if job.get('covalent_bonds'):
            for bond in job['covalent_bonds']:
                covalent_bonds_json.append({
                    "atom1": bond['atom1'],
                    "atom2": bond['atom2']
                })

        job_json = {
            "name": job['name'],
            "sequences": sequences_json,
            "covalent_bonds": covalent_bonds_json
        }

        return json.dumps([job_json], indent=2)

    def _process_job(self, row: pd.Series) -> Tuple[Optional[Dict], List[str]]:
        """Process a single job row from CSV"""
        errors = []
        all_sequences = []

        jobname, jobname_errors = self._sanitize_jobname(row.get('jobname', ''))
        errors.extend(jobname_errors)

        if not jobname:
            return None, errors

        pocket_binder = row.get('pocket_binder', '')
        pocket_contacts = row.get('pocket_contacts', '')
        covalent_bonds_str = row.get('covalent_bonds', '')

        chain_id_counter = 0
        name_to_chain_mapping = {}

        for i in range(1, 11):
            name_col = f'seq{i}_name'
            type_col = f'seq{i}_type'
            copies_col = f'seq{i}_copies'
            seq_col = f'seq{i}'
            mods_col = f'seq{i}_mods'

            if name_col not in row or type_col not in row or seq_col not in row:
                continue

            user_name = row.get(name_col, '')
            seq_type = row.get(type_col, '')
            seq_type = str(seq_type).strip().lower()
            copies = row.get(copies_col, 1)
            sequence = row.get(seq_col, '')
            mods = row.get(mods_col, '')

            if pd.isna(sequence) or str(sequence).strip() == '':
                continue

            sequence = str(sequence).strip()
            copies = int(copies) if pd.notna(copies) and str(copies).strip() != '' else 1
            user_name = str(user_name).strip() if pd.notna(user_name) else ''

            chain_ids = []
            for copy_num in range(copies):
                chain_id = chr(ord('A') + chain_id_counter)
                chain_ids.append(chain_id)

                if user_name and copy_num == 0:
                    name_to_chain_mapping[user_name] = chain_id

                chain_id_counter += 1

            if seq_type.lower() in ['protein', 'dna', 'rna']:
                char_errors = self._validate_sequence_characters(sequence, seq_type)
                errors.extend(char_errors)

                for idx, chain_id in enumerate(chain_ids):
                    remapped_mods = self._remap_modification_chains(mods, name_to_chain_mapping)
                    mods_list, mod_errors = self._parse_modifications(remapped_mods, sequence, chain_id, seq_type)
                    errors.extend(mod_errors)

                    seq_dict = {
                        'type': seq_type.lower(),
                        'id': chain_id,
                        'name': user_name,
                        'sequence': sequence,
                        'modifications': mods_list if mods_list else None
                    }
                    all_sequences.append(seq_dict)

            elif seq_type.lower() == 'ligand':
                ligand_string = sequence.strip()

                # Check CCD lookup FIRST (before SMILES detection)
                looked_up_smiles = None
                is_known_ccd = False
                if ligand_string in self.ligand_lookup:
                    looked_up_smiles = self.ligand_lookup[ligand_string].get('smiles')
                    is_known_ccd = True
                elif ligand_string in self.ion_lookup:
                    looked_up_smiles = self.ion_lookup[ligand_string].get('smiles')
                    is_known_ccd = True

                if not is_known_ccd and not self._is_smiles(ligand_string):
                    errors.append(f"Unknown ligand/ion: '{ligand_string}'")

                for chain_id in chain_ids:
                    if is_known_ccd:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'name': user_name,
                            'ccd': ligand_string
                        }
                    else:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'name': user_name,
                            'smiles': ligand_string
                        }
                    all_sequences.append(seq_dict)

            else:
                errors.append(f"Unsupported sequence type: {seq_type}")

        if not all_sequences:
            errors.append("No valid sequences found")
            return None, errors

        remapped_pocket_binder = name_to_chain_mapping.get(pocket_binder, pocket_binder) if pocket_binder else pocket_binder
        remapped_contacts = self._remap_contact_chains(pocket_contacts, name_to_chain_mapping)

        pocket, pocket_errors = self._parse_pocket(remapped_pocket_binder, remapped_contacts)
        errors.extend(pocket_errors)

        remapped_bonds = self._remap_bond_chains(covalent_bonds_str, name_to_chain_mapping)
        covalent_bonds, bond_errors = self._parse_covalent_bonds(remapped_bonds)
        errors.extend(bond_errors)

        has_modifications = any(seq.get('modifications') for seq in all_sequences)

        job = {
            'name': jobname,
            'sequences': all_sequences,
            'pocket': pocket,
            'covalent_bonds': covalent_bonds,
            'has_modifications': has_modifications,
            'has_pocket': pocket is not None,
            'has_covalent': len(covalent_bonds) > 0
        }

        return job, errors

    def process_csv(self, csv_path: str) -> Tuple[List[Dict], pd.DataFrame]:
        """Process CSV file and return jobs list and summary DataFrame"""
        df = pd.read_csv(csv_path)

        jobs = []
        summary_rows = []

        for idx, row in df.iterrows():
            job, errors = self._process_job(row)

            if job:
                jobs.append(job)
                summary_rows.append({
                    'jobname': job['name'],
                    'sequences': len(job['sequences']),
                    'has_modifications': job['has_modifications'],
                    'has_pocket': job['has_pocket'],
                    'has_covalent': job['has_covalent'],
                    'status': 'ERROR' if errors else 'OK',
                    'errors': '; '.join(errors) if errors else ''
                })
            else:
                summary_rows.append({
                    'jobname': f"Row {idx+1}",
                    'sequences': 0,
                    'has_modifications': False,
                    'has_pocket': False,
                    'has_covalent': False,
                    'status': 'FAILED',
                    'errors': '; '.join(errors)
                })

        summary_df = pd.DataFrame(summary_rows)
        return jobs, summary_df

print("Initializing Protenix CSV Processor...")
protenix_processor = ProtenixJobProcessor()

print("Using embedded reference data: 80 entries")
print("Processor ready")
print("Reference data includes:")
print(f"   - 16 PTM types")
print(f"   - 24 ligand types")
print(f"   - 11 ion types")
print(f"   - 10 glycan types")
print(f"   - 8 DNA modification types")
print(f"   - 10 RNA modification types")
print(f"\nUsing embedded reference data (common PTMs, ligands, ions, glycans, DNA/RNA mods)")
print("\nNote: Chain IDs are assigned as A, B, C, D... sequentially")
print("   Sequence identity is preserved in job names")
print("   Protenix uses AF3-like JSON format (CSV is auto-converted)")



# ============================================================
# PROCESS UPLOADED CSV
# ============================================================
import pandas as pd

if 'global_settings' not in globals() or 'csv_filename' not in global_settings:
    print("Error: Please run Cell 2 (Upload CSV) first")
    raise ValueError("No CSV uploaded")

csv_filename = global_settings['csv_filename']

# Initialize processor
try:
    protenix_processor = ProtenixJobProcessor()
except Exception as e:
    print(f"Failed to initialize processor: {e}")
    raise

print("\nProcessing CSV...")
try:
    jobs, summary_df = protenix_processor.process_csv(csv_filename)
except Exception as e:
    print(f"CSV processing failed: {e}")
    raise

print("\n" + "=" * 60)
print("JOB SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

if jobs:
    print("\n" + "=" * 60)
    print("JSON PREVIEW (first job)")
    print("=" * 60)
    preview_json = protenix_processor._generate_json(jobs[0])
    print(preview_json)

error_jobs = summary_df[summary_df['status'] == 'ERROR']
if len(error_jobs) > 0:
    print(f"\n{len(error_jobs)} jobs have errors:")
    for _, row in error_jobs.iterrows():
        print(f"  - {row['jobname']}: {row['errors']}")

    proceed = input("\nProceed with valid jobs only? (yes/no): ").strip().lower()
    if proceed not in ['yes', 'y']:
        raise ValueError("Processing cancelled by user")

global_settings.update({
    'batch_jobs': jobs,
    'processor': protenix_processor,
})

print("\n" + "=" * 60)
print(f"READY TO PROCESS {len(jobs)} JOBS")
print("=" * 60)
if global_settings.get('msa_folder'):
    print(f"Pre-computed MSAs: {global_settings['msa_folder']}")

print("\nNext: Configure settings (Cell 5), then run (Cell 6)")

In [ ]:
#@title Cell 5: Configure Protenix Prediction Parameters

print("=" * 60)
print("PROTENIX PREDICTION PARAMETERS")
print("=" * 60)

# Model Selection
model_name = "protenix_base_20250630_v1.0.0" #@param ["protenix_base_20250630_v1.0.0", "protenix_base_default_v1.0.0", "protenix_base_default_v0.5.0", "protenix_base_constraint_v0.5.0", "protenix_mini_esm_v0.5.0", "protenix_mini_default_v0.5.0", "protenix_mini_ism_v0.5.0", "protenix_tiny_default_v0.5.0"]
#@markdown - **Model**: protenix_base (highest accuracy), protenix_mini (faster, less VRAM). ESM variant uses ESM embeddings. **VRAM**: base ~15GB, mini ~8GB. **Time**: mini ~2x faster.

# Seeds
seeds = "101" #@param {type:"string"}
#@markdown - **Random seed(s)**: Single int (e.g., "101") or comma-separated for multiple runs (e.g., "101,102,103"). Controls reproducibility. **Time**: Each seed runs a full prediction. **VRAM**: No impact per seed.

# Sampling & Recycling
diffusion_samples = 5 #@param {type:"integer"}
#@markdown - **Diffusion samples**: Number of structure samples per seed. More = better chance of good structure. **Time**: Linear. **VRAM**: Constant.

diffusion_steps = 200 #@param {type:"integer"}
#@markdown - **Diffusion steps**: Denoising steps per sample. 200 is default for base models. **Time**: Linear.

pairformer_cycles = 10 #@param {type:"integer"}
#@markdown - **Pairformer cycles**: Recycling iterations. 10 is default for base models. **Time**: Linear.

# MSA Settings
use_msa = True #@param {type:"boolean"}
#@markdown - **Use MSA**: Enable Multiple Sequence Alignment for better accuracy. Adds 30-120s per protein sequence but significantly improves predictions. **Time**: +30-120s per protein. **VRAM**: Minimal impact.


use_rna_msa = False #@param {type:"boolean"}
#@markdown - **Use RNA MSA**: Enable MSA generation for RNA sequences. Experimental feature, may not always improve results. **Time**: +30-120s per RNA. **VRAM**: Minimal impact.

msa_server_mode = "protenix" #@param ["protenix", "colabfold"]
#@markdown - **MSA server backend**: `protenix` uses built-in MSA pipeline, `colabfold` uses the ColabFold API. **Time**: Similar. **VRAM**: No impact.

# Computation Settings
dtype = "bf16" #@param ["bf16", "fp32", "fp16"]
#@markdown - **Computation precision**: bf16 (bfloat16) is recommended for best speed/accuracy balance. fp32 for maximum precision (slower, more VRAM). **Time**: bf16 ~2x faster than fp32. **VRAM**: bf16 ~50% less than fp32.

enable_cache = True #@param {type:"boolean"}
#@markdown - **Enable cache**: Cache intermediate computations for faster inference. Recommended unless running out of memory. **Time**: Faster with cache. **VRAM**: Slightly more with cache enabled.

enable_fusion = True #@param {type:"boolean"}
#@markdown - **Enable fusion**: Enable kernel fusion optimizations. Recommended for better GPU utilization. **Time**: Faster with fusion. **VRAM**: Slightly less with fusion.

skip_existing = False  #@param {type:"boolean"}
#@markdown - **Skip existing**: If True, skip jobs that already have output in their base directory. Useful for resuming interrupted batches.

# Output Info
#@markdown ---
#@markdown **Output format**: Protenix always outputs mmCIF (.cif) files. Per-atom pLDDT and PAE matrices are included automatically.

#@markdown ---
#@markdown ### Interface Analysis (ipSAE_batch)
#@markdown Scores interface quality (ipSAE, ipTM, pDockQ) and generates visualizations for each prediction.

enable_ipsae = True #@param {type:"boolean"}
#@markdown - **Enable**: Run ipSAE interface analysis on each completed job.

ipsae_png = True #@param {type:"boolean"}
#@markdown - **PNG plots**: Matrix + ribbon visualizations per model (~10-30s/job).

ipsae_pdf = False #@param {type:"boolean"}
#@markdown - **PDF report**: Side-by-side model comparison per job (~20-60s/job).

ipsae_per_residue = False #@param {type:"boolean"}
#@markdown - **Per-residue CSV**: Detailed per-residue interface scores.

ipsae_per_contact = False #@param {type:"boolean"}
#@markdown - **Per-contact CSV**: Per-contact-pair scores (most detailed).

ipsae_pae_cutoff = 10.0 #@param {type:"number"}
#@markdown - **PAE cutoff**: PAE threshold for interface scoring (default: 10.0).

ipsae_dist_cutoff = 10.0 #@param {type:"number"}
#@markdown - **Distance cutoff**: CB-CB distance threshold in Angstroms (default: 10.0).

ipsae_select_residues = "" #@param {type:"string"}
#@markdown - **Select residues**: Focus on specific residues, e.g. `A:100-105,B:203`. Empty = all interfaces.

ipsae_batch_analysis = True #@param {type:"boolean"}
#@markdown - **Final batch comparison**: Combined CSV + interactive HTML across all jobs.

# Check if global_settings exists
if 'global_settings' not in globals():
    print("\nError: Please run Cell 2 (Upload CSV) first")
    raise ValueError("No batch jobs configured")

# Store all settings
global_settings['config'] = {
    'model_name': model_name,
    'seeds': seeds,
    'diffusion_samples': diffusion_samples,
    'diffusion_steps': diffusion_steps,
    'pairformer_cycles': pairformer_cycles,
    'use_msa': use_msa,
    'use_rna_msa': use_rna_msa,
    'msa_server_mode': msa_server_mode,
    'dtype': dtype,
    'enable_cache': enable_cache,
    'enable_fusion': enable_fusion,
    'skip_existing': skip_existing,
    'enable_ipsae': enable_ipsae,
    'ipsae_png': ipsae_png,
    'ipsae_pdf': ipsae_pdf,
    'ipsae_per_residue': ipsae_per_residue,
    'ipsae_per_contact': ipsae_per_contact,
    'ipsae_pae_cutoff': ipsae_pae_cutoff,
    'ipsae_dist_cutoff': ipsae_dist_cutoff,
    'ipsae_select_residues': ipsae_select_residues,
    'ipsae_batch_analysis': ipsae_batch_analysis,
}

# Display summary
print("\n" + "=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"Model Settings:")
print(f"  - Model: {model_name}")
print(f"  - Seeds: {seeds}")
print(f"\nMSA Settings:")
print(f"  - Use MSA: {use_msa}")
if global_settings.get('msa_folder'):
    print(f"  - PRE-COMPUTED MSAs: {global_settings['msa_folder']}")
else:
    print(f"  - MSA source: online ({msa_server_mode})")
print(f"  - Use RNA MSA: {use_rna_msa}")
print(f"\nComputation:")
print(f"  - Precision (dtype): {dtype}")
print(f"  - Enable cache: {enable_cache}")
print(f"  - Enable fusion: {enable_fusion}")
print(f"\nOutput:")

print(f"\nInterface Analysis (ipSAE):")
print(f"  - Enabled: {enable_ipsae}")
if enable_ipsae:
    print(f"  - PNG: {ipsae_png} | PDF: {ipsae_pdf}")
    print(f"  - Per-residue: {ipsae_per_residue} | Per-contact: {ipsae_per_contact}")
    print(f"  - Cutoffs: PAE={ipsae_pae_cutoff}, dist={ipsae_dist_cutoff}")
    if ipsae_select_residues:
        print(f"  - Selected residues: {ipsae_select_residues}")
    print(f"  - Final batch comparison: {ipsae_batch_analysis}")

# Model size warning
if 'mini' in model_name:
    print(f"\nNote: Using mini model - faster but lower accuracy than base model")
elif 'esm' in model_name:
    print(f"\nNote: Using ESM embeddings - may improve accuracy for some targets")

print(f"\nNext: Run Cell 6 to start batch predictions")


In [ ]:
#@title Cell 6: Run Batch Predictions with Calibration-Based Parallel GPU Scheduling
#@markdown **Runs job 1 alone to calibrate VRAM usage, then launches remaining jobs in parallel when GPU memory allows.**
#@markdown Results are automatically uploaded to Google Drive after each job completes.
%%time

import subprocess
import os
import zipfile
import shutil

def get_unique_job_dir(job_name):
    """Return a unique job directory, appending _2, _3, etc. if exists."""
    if not os.path.exists(job_name):
        return job_name
    n = 2
    while os.path.exists(f"{job_name}_{n}"):
        n += 1
    return f"{job_name}_{n}"

def has_existing_output(job_name, search_dir="/content"):
    """Return True if job_name/ contains any .cif or .pdb structure files."""
    job_dir = os.path.join(search_dir, job_name)
    if not os.path.isdir(job_dir):
        return False
    for root, _, files in os.walk(job_dir):
        for f in files:
            if f.endswith(('.cif', '.pdb')):
                return True
    return False

import time
import gc
import threading
import queue
import re
import json
from datetime import datetime


# ============================================================
# VERIFY SETUP
# ============================================================
if 'global_settings' not in globals():
    print("Error: Please run Cells 2-5 first")
    raise ValueError("Configuration not found")

if not global_settings.get('batch_jobs'):
    print("Error: No batch jobs configured")
    print("   Please run Cell 2 to upload and process your CSV")
    raise ValueError("No jobs to process")

config = global_settings['config']
jobs = global_settings['batch_jobs']
processor = global_settings['processor']
drive = global_settings.get('drive')
gdrive_folder_name = global_settings.get('gdrive_folder_name', 'Protenix_Results')
msa_folder = global_settings.get('msa_folder', '')

# Check ipSAE_batch availability
ipsae_available = False
try:
    result = subprocess.run(["ipsae-batch", "--help"], capture_output=True, text=True, timeout=10)
    ipsae_available = (result.returncode == 0)
except Exception:
    pass
if config.get('enable_ipsae') and not ipsae_available:
    print("WARNING: ipSAE_batch not installed. Interface analysis will be skipped.")

# ============================================================
# PARALLEL EXECUTION SETTINGS
# ============================================================
enable_parallel = True #@param {type:"boolean"}
#@markdown - **Enable parallel execution**: Launch multiple jobs simultaneously when VRAM allows
#@markdown > **Note**: Jobs are automatically sorted largest-first for optimal calibration. The largest job calibrates VRAM, giving the most accurate per-token rate for parallel scheduling.

vram_limit = 0.85 #@param {type:"slider", min:0.6, max:0.95, step:0.05}
#@markdown - **VRAM limit**: Max fraction of GPU memory to use (0.85 = 85%). Jobs won't launch if this would be exceeded.

# ============================================================
# GPU VERIFICATION
# ============================================================
print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)
gpu_available = False
total_gpu_vram = 0
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        total_gpu_vram = gpu_memory_gb
        print(f"GPU: {gpu_name}")
        print(f"   Total Memory: {gpu_memory_gb:.1f} GB")
        print(f"   VRAM limit ({vram_limit*100:.0f}%): {gpu_memory_gb * vram_limit:.1f} GB")

        test_tensor = torch.zeros(1).cuda()
        del test_tensor
        torch.cuda.synchronize()
        gpu_available = True

        try:
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=2)
            if result.returncode == 0:
                initial_mem = float(result.stdout.strip()) / 1024
                print(f"   Current Usage: {initial_mem:.2f} GB")
                print(f"   GPU monitoring ready")
            else:
                print(f"   nvidia-smi not available, parallel mode disabled")
                enable_parallel = False
        except Exception as e:
            print(f"   nvidia-smi test failed: {e}")
            enable_parallel = False
    else:
        print("WARNING: No GPU detected")
        print("   Predictions will be very slow on CPU")
        enable_parallel = False
except Exception as e:
    print(f"GPU test failed: {e}")
    gpu_available = False
    enable_parallel = False

# Precision check
if config.get('dtype') == 'bf16':
    print(f"\nUsing BF16 precision - good balance of speed and stability")
else:
    print(f"\nUsing {config.get('dtype')} precision")

# ============================================================
# GPU MONITOR CLASS
# ============================================================
class GPUMonitor:
    """Monitor GPU memory usage using nvidia-smi"""

    def __init__(self):
        self.monitoring = False
        self.peak_memory = 0
        self.current_memory = 0
        self.monitor_thread = None
        self._lock = threading.Lock()
        self.gpu_available = self._test_nvidia_smi()
        self._pid_registry = {}  # shell_pid -> {name, peak_gb}

    def _test_nvidia_smi(self):
        try:
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=2)
            return result.returncode == 0
        except:
            return False

    def _get_gpu_memory(self):
        if not self.gpu_available:
            return 0
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=2
            )
            if result.returncode == 0:
                return float(result.stdout.strip()) / 1024  # MB to GB
        except:
            pass
        return 0

    def _monitor_loop(self):
        while self.monitoring:
            mem = self._get_gpu_memory()
            # Per-PID VRAM tracking
            job_mem = {}
            with self._lock:
                need_pid = bool(self._pid_registry)
                shells = set(self._pid_registry.keys()) if need_pid else set()
            if need_pid:
                pid_mem = self._get_pid_mem()
                for gpu_pid, gb in pid_mem.items():
                    shell = self._find_job_for_gpu_pid(gpu_pid, shells)
                    if shell is not None:
                        job_mem[shell] = job_mem.get(shell, 0) + gb
            with self._lock:
                self.current_memory = mem
                if mem > self.peak_memory:
                    self.peak_memory = mem
                for spid, cur_gb in job_mem.items():
                    if spid in self._pid_registry:
                        if cur_gb > self._pid_registry[spid]['peak_gb']:
                            self._pid_registry[spid]['peak_gb'] = cur_gb
            time.sleep(0.5)

    def start(self):
        if not self.gpu_available:
            return
        with self._lock:
            self.monitoring = True
            self.peak_memory = 0
            self.current_memory = 0
        self.monitor_thread = threading.Thread(target=self._monitor_loop, daemon=True)
        self.monitor_thread.start()

    def stop(self):
        self.monitoring = False
        if self.monitor_thread:
            self.monitor_thread.join(timeout=2)
        return self.peak_memory

    def get_current(self):
        with self._lock:
            return self.current_memory

    def get_peak(self):
        with self._lock:
            return self.peak_memory

    def reset_peak(self):
        with self._lock:
            self.peak_memory = self.current_memory

    def _get_pid_mem(self):
        """Get per-PID GPU memory {pid: GB}."""
        try:
            r = subprocess.run(
                ['nvidia-smi', '--query-compute-apps=pid,used_gpu_memory', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=2
            )
            if r.returncode == 0 and r.stdout.strip():
                result = {}
                for ln in r.stdout.strip().split('\n'):
                    parts = ln.strip().split(',')
                    if len(parts) >= 2:
                        try:
                            result[int(parts[0].strip())] = float(parts[1].strip()) / 1024
                        except ValueError:
                            pass
                return result
        except:
            pass
        return {}

    def _find_job_for_gpu_pid(self, gpu_pid, registered_shells):
        """Walk process tree up to find which registered job owns this GPU PID."""
        current = gpu_pid
        visited = set()
        while current > 1 and current not in visited:
            visited.add(current)
            if current in registered_shells:
                return current
            try:
                with open(f'/proc/{current}/status') as f:
                    for line in f:
                        if line.startswith('PPid:'):
                            current = int(line.split(':')[1].strip())
                            break
                    else:
                        break
            except (FileNotFoundError, PermissionError, ValueError):
                break
        return None

    def register_pid(self, shell_pid, job_name):
        """Register a job process PID for per-job VRAM tracking."""
        with self._lock:
            self._pid_registry[shell_pid] = {'name': job_name, 'peak_gb': 0.0}

    def unregister_pid(self, shell_pid):
        """Unregister and return peak VRAM (GB) for a job."""
        with self._lock:
            entry = self._pid_registry.pop(shell_pid, None)
            return entry['peak_gb'] if entry else 0.0

    def get_pid_peak(self, shell_pid):
        """Get peak VRAM (GB) for a specific registered job."""
        with self._lock:
            entry = self._pid_registry.get(shell_pid)
            return entry['peak_gb'] if entry else 0.0


gpu_monitor = GPUMonitor()

def clear_gpu_cache():
    if not gpu_available:
        return
    try:
        import torch
        torch.cuda.empty_cache()
        gc.collect()
    except:
        pass

# ============================================================
# GOOGLE DRIVE HELPERS
# ============================================================
def get_or_create_folder(drive, folder_name, parent_id='root'):
    if not drive:
        return None
    try:
        file_list = drive.ListFile({
            'q': f"'{parent_id}' in parents and title='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        }).GetList()
        if file_list:
            print(f"Found existing folder: {folder_name}")
            return file_list[0]['id']
        else:
            folder = drive.CreateFile({
                'title': folder_name,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [{'id': parent_id}]
            })
            folder.Upload()
            print(f"Created new folder: {folder_name}")
            return folder['id']
    except Exception as e:
        print(f"Error with folder '{folder_name}': {e}")
        return None

def upload_to_gdrive(drive, file_path, folder_id, job_name):
    if not drive or not os.path.exists(file_path):
        return None
    try:
        uploaded_file = drive.CreateFile({
            'title': os.path.basename(file_path),
            'parents': [{'id': folder_id}]
        })
        uploaded_file.SetContentFile(file_path)
        uploaded_file.Upload()
        file_url = f"https://drive.google.com/file/d/{uploaded_file['id']}/view"
        print(f"  Uploaded to Google Drive: {file_url}")
        return file_url
    except Exception as e:
        print(f"  Upload failed: {e}")
        return None

# ============================================================
# UTILITY FUNCTIONS
# ============================================================
def read_error_files(job_dir, job_name):
    error_info = []
    error_dirs = [
        os.path.join(job_dir, "errors"),
        os.path.join(job_dir, job_name, "errors"),
    ]
    for error_dir in error_dirs:
        if os.path.exists(error_dir):
            try:
                for f in os.listdir(error_dir):
                    if f.endswith(('.txt', '.log', '.json')):
                        try:
                            with open(os.path.join(error_dir, f), 'r') as fh:
                                error_info.append({'file': f, 'content': fh.read()[:500]})
                        except:
                            pass
            except:
                pass
    return error_info

def find_predictions_directory(job_dir, job_name, seeds_list):
    """Find output directory containing structure files.

    Protenix outputs to: {output_dir}/{job_name}/{seed}/ with .cif files
    """
    possible_paths = [job_dir]

    # Add seed-specific paths
    for seed in seeds_list:
        possible_paths.append(os.path.join(job_dir, job_name, str(seed)))
        possible_paths.append(os.path.join(job_dir, str(seed)))

    # Add generic paths
    possible_paths.extend([
        os.path.join(job_dir, job_name),
        os.path.join(job_dir, "predictions"),
        os.path.join(job_dir, job_name, "predictions"),
        os.path.join(job_dir, "predictions", job_name),
    ])

    for path in possible_paths:
        if os.path.exists(path):
            try:
                for root, dirs, files_list in os.walk(path):
                    for file in files_list:
                        if file.endswith(('.cif', '.pdb', '.mmcif')):
                            return path
            except:
                continue
    return None

def parse_protenix_progress(line):
    keywords = ['progress', 'step', 'recycling', 'diffusion', 'sample',
                'completed', 'saving', 'msa', 'model', 'generating',
                'loading', 'processing', 'predicting', 'running',
                'inference', 'template', 'featuriz', 'writing']
    line_lower = line.lower()
    return any(kw in line_lower for kw in keywords)

def count_job_tokens(job, processor):
    """Count tokens for VRAM estimation.
    Protein/DNA/RNA: len(sequence) + modification atom counts
    Ligand CCD: lookup heavy atom count
    Ligand SMILES: 30 (conservative estimate)
    Ion: 1
    """
    total_tokens = 0
    for seq in job.get('sequences', []):
        seq_type = seq.get('type', '').lower()
        if seq_type in ['protein', 'dna', 'rna']:
            total_tokens += len(seq.get('sequence', ''))
            modifications = seq.get('modifications')
            if modifications:
                for mod in modifications:
                    mod_code = mod.get('ccd', '')
                    if seq_type == 'protein':
                        info = processor.ptm_lookup.get(mod_code) or processor.glycan_lookup.get(mod_code)
                    elif seq_type == 'dna':
                        info = processor.dna_mod_lookup.get(mod_code)
                    elif seq_type == 'rna':
                        info = processor.rna_mod_lookup.get(mod_code)
                    else:
                        info = None
                    if info:
                        total_tokens += info.get('heavy_atom_count', 0)
        elif seq_type == 'ligand':
            ccd_code = seq.get('ccd')
            smiles = seq.get('smiles')
            if ccd_code:
                if ccd_code in processor.ligand_lookup:
                    total_tokens += processor.ligand_lookup[ccd_code].get('heavy_atom_count', 20)
                elif ccd_code in processor.ion_lookup:
                    total_tokens += 1
                elif ccd_code in processor.glycan_lookup:
                    total_tokens += processor.glycan_lookup[ccd_code].get('heavy_atom_count', 15)
                else:
                    total_tokens += 20
            elif smiles:
                total_tokens += 30
    return max(total_tokens, 1)

# ============================================================
# NON-BLOCKING PROCESS OUTPUT READER
# ============================================================
def _reader_thread(pipe, output_queue):
    """Read lines from a pipe into a thread-safe queue."""
    try:
        for line in iter(pipe.readline, ''):
            output_queue.put(line.rstrip('\n'))
    except:
        pass
    finally:
        try:
            pipe.close()
        except:
            pass

# ============================================================
# MSA FOLDER LOOKUP HELPER
# ============================================================
def find_precomputed_msas(job, msa_folder):
    """Look up pre-computed MSA files for ALL protein chains in a job.

    All-or-nothing: returns MSA paths only if EVERY protein chain has a
    pre-computed MSA file. If any chain is missing, returns empty dict
    and the job falls back to online MSA search entirely.

    Handles naming mismatches between the MSA notebook (raw jobname) and
    folding notebooks (sanitized jobname) by normalizing case and punctuation.

    Args:
        job: Job dict with 'name' and 'sequences'
        msa_folder: Path to paired/ MSA folder

    Returns:
        dict: {chain_name: msa_file_path} for ALL protein chains, or {} if any missing
    """
    if not msa_folder or not os.path.isdir(msa_folder):
        return {}

    job_name = job['name']
    found = {}
    missing = []

    # Collect unique protein chain names
    seen = set()
    protein_chains = []
    for seq in job.get('sequences', []):
        if seq.get('type', '').lower() != 'protein':
            continue
        chain_name = seq.get('name', '')
        if not chain_name or chain_name in seen:
            continue
        seen.add(chain_name)
        protein_chains.append(chain_name)

    if not protein_chains:
        return {}

    # Build index of actual files in the directory.
    # Normalize filenames (lowercase + hyphens to underscores) so that
    # MSA notebook output (raw jobname e.g. "Rx-ADP") matches folding
    # notebook lookups (sanitized e.g. "rx_adp").
    available_files = {}
    for fname in os.listdir(msa_folder):
        if fname.endswith('_paired.a3m'):
            normalized = fname.lower().replace('-', '_')
            available_files[normalized] = fname

    for chain_name in protein_chains:
        expected = f"{job_name}_{chain_name}_paired.a3m"
        exact_path = os.path.join(msa_folder, expected)
        if os.path.isfile(exact_path):
            found[chain_name] = exact_path
        else:
            # Normalized lookup (handles case + hyphen/underscore differences)
            norm_key = expected.lower().replace('-', '_')
            if norm_key in available_files:
                found[chain_name] = os.path.join(msa_folder, available_files[norm_key])
            else:
                missing.append(chain_name)

    if missing:
        print(f"   Pre-computed MSAs missing for: {missing} -- using online MSA for all chains")
        return {}

    print(f"   Pre-computed MSAs found for all {len(found)} chain(s): {list(found.keys())}")
    return found


def build_protenix_cmd(json_file, job_dir, current_config, msas_injected=False):
    """Build the protenix pred CLI command."""
    cmd_parts = [
        "protenix", "pred",
        "-i", json_file,
        "-o", job_dir,
        "-n", current_config['model_name'],
        "-s", current_config['seeds'],
        "-e", str(current_config.get('diffusion_samples', 5)),
        "-p", str(current_config.get('diffusion_steps', 200)),
        "-c", str(current_config.get('pairformer_cycles', 10)),
        "--use_msa", "true" if msas_injected else str(current_config['use_msa']).lower(),
        "--use_template", "false",
        "--dtype", current_config['dtype'],
    ]

    if current_config.get('use_rna_msa'):
        cmd_parts.extend(["--use_rna_msa", "true"])

    msa_mode = current_config.get('msa_server_mode', 'colabfold')
    cmd_parts.extend(["--msa_server_mode", msa_mode])

    if current_config.get('enable_cache', True):
        cmd_parts.extend(["--enable_cache", "true"])

    if current_config.get('enable_fusion', True):
        cmd_parts.extend(["--enable_fusion", "true"])

    cmd_parts.extend(["--need_atom_confidence", "True"])

    # Blackwell: use PyTorch fallback for trimul (Triton segfaults on sm_120)
    if global_settings.get('is_blackwell', False):
        cmd_parts.extend(["--trimul_kernel", "torch", "--triatt_kernel", "cuequivariance"])

    return " ".join(cmd_parts)

# ============================================================
# ipSAE INTERFACE ANALYSIS
# ============================================================
def run_ipsae_analysis(job_dir, job_name):
    """Run ipSAE per-job analysis. Returns True on success, False on failure."""
    if not ipsae_available or not config.get('enable_ipsae'):
        return False

    ipsae_output = os.path.join(job_dir, "ipsae_analysis")
    os.makedirs(ipsae_output, exist_ok=True)

    cmd_parts = [
        "ipsae-batch", job_dir,
        "--backend", "protenix",
        "--output_dir", ipsae_output,
        "--pae_cutoff", str(config.get('ipsae_pae_cutoff', 10.0)),
        "--dist_cutoff", str(config.get('ipsae_dist_cutoff', 10.0)),
        "--workers", "1",
        "--cores", "1",
    ]
    if config.get('ipsae_png'):
        cmd_parts.append("--png")
    if config.get('ipsae_pdf'):
        cmd_parts.append("--pdf")
    if config.get('ipsae_per_residue'):
        cmd_parts.append("--per_residue")
    if config.get('ipsae_per_contact'):
        cmd_parts.append("--per_contact")
    if config.get('ipsae_select_residues'):
        cmd_parts.extend(["--select-residues", config['ipsae_select_residues']])

    print(f"   Running ipSAE analysis...")
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            n_files = sum(1 for f in os.listdir(ipsae_output) if not f.startswith('.'))
            print(f"   ipSAE complete: {n_files} output files")
            return True
        else:
            print(f"   ipSAE failed (rc={result.returncode}): {result.stderr[:200]}")
            return False
    except subprocess.TimeoutExpired:
        print(f"   ipSAE timed out (300s)")
        return False
    except Exception as e:
        print(f"   ipSAE error: {e}")
        return False

# ============================================================
# PROCESS COMPLETED JOB (zip + upload + collect results)
# ============================================================
def process_completed_job(job_name, job_dir, json_file, job_time, peak_gpu, drive, gdrive_folder_id, seeds_list, log_lines=None, msa_paths=None):
    """Find outputs, zip, upload to GDrive. Returns result dict."""
    # Write run log
    log_file = os.path.join(job_dir, f"{job_name}_log.txt")
    if log_lines:
        with open(log_file, "w") as lf:
            lf.write("\n".join(log_lines))
        print(f"   Saved run log: {os.path.basename(log_file)} ({len(log_lines)} lines)")
    predictions_dir = find_predictions_directory(job_dir, job_name, seeds_list)

    if not predictions_dir:
        print(f"\nNo structure files (.cif/.pdb/.mmcif) found for {job_name}")
        error_files = read_error_files(job_dir, job_name)
        if error_files:
            for err_info in error_files[:2]:
                print(f"   Error file {err_info['file']}: {err_info['content'][:200]}")
        # Show directory structure
        print(f"   Directory structure of {job_dir}:")
        for root, dirs, files_list in os.walk(job_dir):
            level = root.replace(job_dir, '').count(os.sep)
            indent = '   ' * (level + 1)
            print(f"{indent}{os.path.basename(root)}/")
            for file in files_list[:10]:
                print(f"{indent}  {file}")
        return {
            'success': False, 'name': job_name,
            'error': 'No structure files generated',
            'time': job_time, 'gpu_peak': peak_gpu
        }

    # Collect all structure files recursively
    structure_files = []
    for root, dirs, files_list in os.walk(predictions_dir):
        for f in files_list:
            if f.endswith(('.cif', '.pdb', '.mmcif')):
                structure_files.append(os.path.join(root, f))

    if not structure_files:
        print(f"\nNo structure files in predictions directory for {job_name}")
        return {
            'success': False, 'name': job_name,
            'error': 'No structure files in predictions directory',
            'time': job_time, 'gpu_peak': peak_gpu
        }

    print(f"   {len(structure_files)} structure files generated")
    for f in sorted(structure_files)[:3]:
        print(f"      {os.path.basename(f)}")
    if len(structure_files) > 3:
        print(f"      ... and {len(structure_files) - 3} more")

    # --- ipSAE interface analysis (before zipping) ---
    ipsae_ran = False
    try:
        ipsae_ran = run_ipsae_analysis(job_dir, job_name)
    except Exception as e:
        print(f"   ipSAE error (non-fatal): {e}")

    # Create ZIP (structures + input JSON + MSA alignments)
    zip_filename = f"{job_name}_results.zip"
    zip_path = os.path.join(job_dir, zip_filename)
    include_ext = {'.cif', '.pdb', '.mmcif', '.a3m', '.json'}
    exclude_ext = {'.tar.gz', '.m8', '.sh', '.fasta'}
    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for sf in structure_files:
                arcname = os.path.relpath(sf, job_dir)
                zipf.write(sf, arcname)
            # Add confidence score files
            for root, dirs, files_list in os.walk(predictions_dir):
                for f in files_list:
                    if f.endswith('.json') and ('confidence' in f or 'full_data' in f):
                        fp = os.path.join(root, f)
                        arcname = os.path.relpath(fp, job_dir)
                        zipf.write(fp, arcname)
            # Add MSA alignment files
            for root, dirs, files_list in os.walk(job_dir):
                for f in files_list:
                    if f.endswith('.a3m'):
                        fp = os.path.join(root, f)
                        arcname = os.path.relpath(fp, job_dir)
                        zipf.write(fp, arcname)
            zipf.write(json_file, os.path.basename(json_file))
            # Add environment log for traceability
            if os.path.isfile("/content/environment.txt"):
                zipf.write("/content/environment.txt", "environment.txt")
            # Add run log
            if os.path.isfile(log_file):
                zipf.write(log_file, os.path.basename(log_file))
            # Add pre-computed MSA files used
            if msa_paths:
                for _chain, _path in msa_paths.items():
                    if os.path.isfile(_path):
                        zipf.write(_path, f"msa/{os.path.basename(_path)}")
                    # Also add individual (unpaired) MSA
                    _ind_dir = os.path.dirname(os.path.dirname(_path))
                    _ind_file = os.path.join(_ind_dir, "individual", f"{_chain}.a3m")
                    if os.path.isfile(_ind_file):
                        zipf.write(_ind_file, f"msa/{os.path.basename(_ind_file)}")

            # Add ipSAE analysis files
            ipsae_dir = os.path.join(job_dir, "ipsae_analysis")
            if os.path.isdir(ipsae_dir):
                for root, dirs, files_list in os.walk(ipsae_dir):
                    for f in files_list:
                        fp = os.path.join(root, f)
                        arcname = os.path.join("ipsae_analysis", os.path.relpath(fp, ipsae_dir))
                        zipf.write(fp, arcname)

        print(f"   Created: {zip_filename}")
    except Exception as e:
        print(f"   Failed to create ZIP: {e}")

    # Upload to GDrive
    file_url = None
    if drive and gdrive_folder_id:
        file_url = upload_to_gdrive(drive, zip_path, gdrive_folder_id, job_name)
    else:
        print(f"   Saved locally: {zip_path}")

    return {
        'success': True, 'name': job_name,
        'structures': len(structure_files),
        'time': job_time, 'url': file_url,
        'gpu_peak': peak_gpu,
        'ipsae_ran': ipsae_ran,
    }

# ============================================================
# SEQUENTIAL RUNNER
# ============================================================
def run_single_prediction_sequential(job, job_idx, total_jobs, retry_with_safer=False):
    """Run a single Protenix prediction sequentially with GPU monitoring."""
    job_start = time.time()

    print("\n" + "=" * 60)
    print(f"Job {job_idx}/{total_jobs}: {job['name']}")
    if retry_with_safer:
        print("   RETRY with safer precision settings")
    print("=" * 60)

    # Skip existing check
    if config.get('skip_existing', False) and has_existing_output(job['name']):
        print(f"  \u23ed Skipping {job['name']}: output already exists")
        return {'success': True, 'name': job['name'], 'skipped': True,
                'structures': 0, 'time': 0, 'gpu_peak': 0,
                'tokens': count_job_tokens(job, processor)}

    gpu_monitor.start()
    time.sleep(0.5)
    initial_mem = gpu_monitor.get_current()
    print(f"GPU Memory (start): {initial_mem:.2f} GB")

    job_name = job['name']
    job_dir = get_unique_job_dir(job_name)
    os.makedirs(job_dir, exist_ok=True)


    # Generate JSON input
    json_content = processor._generate_json(job)

    # Inject pre-computed MSA paths (all-or-nothing)
    msas_injected = False
    precomp_msas = {}
    msa_folder_val = msa_folder
    if msa_folder_val:
        precomp_msas = find_precomputed_msas(job, msa_folder_val)
        if precomp_msas:
            import json as _json_msa
            seq_to_name = {}
            for seq in job.get('sequences', []):
                if seq.get('type', '').lower() == 'protein' and seq.get('name'):
                    seq_to_name[seq['sequence']] = seq['name']
            json_data = _json_msa.loads(json_content)
            for item in json_data:
                for seq_entry in item.get('sequences', []):
                    if 'proteinChain' not in seq_entry:
                        continue
                    pc = seq_entry['proteinChain']
                    chain_name = seq_to_name.get(pc.get('sequence', ''))
                    if chain_name and chain_name in precomp_msas:
                        msa_path = precomp_msas[chain_name]
                        pc['pairedMsaPath'] = msa_path
                        ind_dir = os.path.join(os.path.dirname(msa_folder_val), 'individual')
                        ind_file = os.path.join(ind_dir, f"{chain_name}.a3m")
                        pc['unpairedMsaPath'] = ind_file if os.path.isfile(ind_file) else msa_path
            json_content = _json_msa.dumps(json_data, indent=2)
            msas_injected = True
            # Verify MSA paths exist (Protenix falls back to online search if not)
            for _chain_name, _msa_path in precomp_msas.items():
                _exists_p = os.path.isfile(_msa_path)
                _ind_dir = os.path.join(os.path.dirname(msa_folder_val), 'individual')
                _ind_file = os.path.join(_ind_dir, f"{_chain_name}.a3m")
                _exists_u = os.path.isfile(_ind_file)
                print(f"   MSA verify: {_chain_name}")
                print(f"     paired:   {_msa_path} [{'OK' if _exists_p else 'MISSING'}]")
                print(f"     unpaired: {_ind_file} [{'OK' if _exists_u else 'MISSING'}]")
    json_file = os.path.join(job_dir, f"{job_name}.json")
    with open(json_file, 'w') as f:
        f.write(json_content)

    current_config = config.copy()
    if retry_with_safer and current_config.get('dtype') == 'fp32':
        current_config['dtype'] = 'bf16'
        print(f"   Changed precision: fp32 -> bf16")

    cmd = build_protenix_cmd(json_file, job_dir, current_config, msas_injected=msas_injected)
    print(f"Command: {cmd}")
    print(f"\nRunning Protenix prediction...")

    # Parse seeds list for output directory detection
    seeds_list = [s.strip() for s in current_config['seeds'].split(',')]

    prediction_failed = False
    error_messages = []
    all_log_lines = []

    # Phase tracking
    phase_times = {}
    current_phase = "initialization"
    phase_start = time.time()

    try:
        process = subprocess.Popen(
            cmd, shell=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True
        )

        # Non-blocking reader
        output_q = queue.Queue()
        reader_t = threading.Thread(target=_reader_thread, args=(process.stdout, output_q), daemon=True)
        reader_t.start()

        print("\nProgress:")
        while process.poll() is None:
            while not output_q.empty():
                try:
                    line = output_q.get_nowait()
                except queue.Empty:
                    break
                if not line:
                    continue
                all_log_lines.append(line)

                # Track errors
                if 'error' in line.lower() and ('nan' in line.lower() or 'inf' in line.lower()):
                    error_messages.append(line)
                    prediction_failed = True
                elif 'out of memory' in line.lower() or 'oom' in line.lower():
                    error_messages.append(line)
                    prediction_failed = True

                # Track phases
                ll = line.lower()
                if 'msa' in ll and current_phase != "msa":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "msa"; phase_start = time.time()
                elif 'template' in ll and current_phase != "template":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "template"; phase_start = time.time()
                elif ('recycling' in ll or 'inference' in ll) and current_phase != "inference":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "inference"; phase_start = time.time()
                elif 'diffusion' in ll and current_phase != "diffusion":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "diffusion"; phase_start = time.time()
                elif ('saving' in ll or 'writing' in ll) and current_phase != "saving":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "saving"; phase_start = time.time()

                # Print progress
                if parse_protenix_progress(line):
                    cur_gpu = gpu_monitor.get_current()
                    if cur_gpu > 1.0:
                        print(f"   {line[:100]} [GPU: {cur_gpu:.1f} GB]")
                    else:
                        print(f"   {line[:100]}")
                elif 'error' in line.lower() or 'warning' in line.lower():
                    print(f"   WARNING: {line[:100]}")

            time.sleep(0.5)

        # Drain remaining output
        while not output_q.empty():
            try:
                line = output_q.get_nowait()
                if line:
                    all_log_lines.append(line)
                    if parse_protenix_progress(line) or 'error' in line.lower():
                        print(f"   {line[:100]}")
            except queue.Empty:
                break

        phase_times[current_phase] = time.time() - phase_start
        return_code = process.wait(timeout=60)
        peak_gpu_memory = gpu_monitor.stop()

        print(f"\nGPU Memory: Peak={peak_gpu_memory:.2f} GB, Final={gpu_monitor.get_current():.2f} GB")

        if return_code != 0:
            print(f"\nPrediction failed (return code: {return_code})")
            error_files = read_error_files(job_dir, job_name)
            if error_files:
                for err_info in error_files[:2]:
                    print(f"   {err_info['file']}: {err_info['content'][:200]}")
            return {
                'success': False, 'name': job_name,
                'error': f"Process exited with code {return_code}",
                'error_messages': error_messages,
                'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory,
                'can_retry': prediction_failed and not retry_with_safer
            }

    except subprocess.TimeoutExpired:
        process.kill()
        peak_gpu_memory = gpu_monitor.stop()
        return {
            'success': False, 'name': job_name, 'error': "Timeout",
            'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory, 'can_retry': False
        }
    except Exception as e:
        peak_gpu_memory = gpu_monitor.stop()
        return {
            'success': False, 'name': job_name, 'error': str(e),
            'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory, 'can_retry': False
        }

    # Print timing
    if phase_times:
        total_phase_time = sum(phase_times.values())
        print(f"\nTiming:")
        for phase, duration in phase_times.items():
            if duration > 0.1:
                pct = (duration / total_phase_time) * 100 if total_phase_time > 0 else 0
                print(f"   - {phase.capitalize()}: {duration:.1f}s ({pct:.0f}%)")

    job_time = time.time() - job_start
    result = process_completed_job(job_name, job_dir, json_file, job_time, peak_gpu_memory, drive, gdrive_folder_id, seeds_list, log_lines=all_log_lines, msa_paths=precomp_msas if msas_injected else None)
    result['can_retry'] = prediction_failed and not retry_with_safer
    if 'tokens' not in result:
        result['tokens'] = count_job_tokens(job, processor)
    print(f"\nTotal: {job_time:.1f}s ({job_time/60:.1f} min)")
    return result

# ============================================================
# SETUP GOOGLE DRIVE
# ============================================================
gdrive_folder_id = None
if drive:
    print("\n" + "=" * 60)
    print("SETTING UP GOOGLE DRIVE")
    print("=" * 60)
    gdrive_folder_id = get_or_create_folder(drive, gdrive_folder_name)
    if gdrive_folder_id:
        print(f"Results will be uploaded to: {gdrive_folder_name}")
    else:
        print("Could not setup Drive folder - results will be saved locally only")
        drive = None
else:
    print("\n" + "=" * 60)
    print("NO GOOGLE DRIVE - LOCAL ONLY")
    print("=" * 60)
    print("Google Drive not configured in Cell 3")

# ============================================================
# PRINT BATCH INFO
# ============================================================
print("\n" + "=" * 60)
print(f"STARTING BATCH PROCESSING: {len(jobs)} JOBS")
print("=" * 60)
print(f"Configuration:")
print(f"  - Model: {config['model_name']}")
print(f"  - Seeds: {config['seeds']}")
print(f"  - Samples: {config.get('diffusion_samples', 5)}")
print(f"  - Diffusion steps: {config.get('diffusion_steps', 200)}")
print(f"  - Pairformer cycles: {config.get('pairformer_cycles', 10)}")
print(f"  - Use MSA: {config['use_msa']}")
print(f"  - Precision: {config['dtype']}")
print(f"  - Cache: {config['enable_cache']}")
print(f"  - Fusion: {config['enable_fusion']}")
print(f"  - Parallel mode: {enable_parallel}")
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Sort jobs largest-first for optimal calibration accuracy
jobs = sorted(jobs, key=lambda j: count_job_tokens(j, processor), reverse=True)
print(f"Jobs sorted by size (largest first):")
for _j in jobs:
    print(f"  {_j['name']}: {count_job_tokens(_j, processor)} tokens")

# ============================================================
# SINGLE JOB SHORTCUT
# ============================================================
if len(jobs) == 1:
    enable_parallel = False

# ============================================================
# MAIN EXECUTION
# ============================================================
batch_start_time = time.time()
completed_jobs = []
failed_jobs = []
retried_jobs = []

if not enable_parallel or not gpu_available:
    # --------------------------------------------------------
    # SEQUENTIAL MODE
    # --------------------------------------------------------
    if not enable_parallel:
        print(f"\nRunning in SEQUENTIAL mode (parallel disabled)")
    else:
        print(f"\nRunning in SEQUENTIAL mode (no GPU monitoring)")

    for job_idx, job in enumerate(jobs, 1):
        result = run_single_prediction_sequential(job, job_idx, len(jobs))

        if result['success']:
            completed_jobs.append(result)
        else:
            if result.get('can_retry'):
                print(f"\nRetrying {job['name']} with safer precision...")
                retried_jobs.append(job['name'])
                clear_gpu_cache()
                time.sleep(2)
                result = run_single_prediction_sequential(job, job_idx, len(jobs), retry_with_safer=True)
                if result['success']:
                    completed_jobs.append(result)
                else:
                    failed_jobs.append(result)
            else:
                failed_jobs.append(result)

        clear_gpu_cache()
        time.sleep(1)

else:
    # --------------------------------------------------------
    # CALIBRATION-BASED PARALLEL MODE
    # --------------------------------------------------------
    print(f"\n" + "=" * 60)
    print(f"CALIBRATION-BASED PARALLEL MODE")
    print("=" * 60)
    print(f"Phase 1: Run job 1 alone to measure actual VRAM usage")
    print(f"Phase 2: Use measurement to estimate remaining jobs")
    print(f"Phase 3: Launch jobs in parallel when VRAM allows")
    print("=" * 60)

    # --- PHASE 1: CALIBRATION (try jobs one by one until one succeeds) ---
    print(f"\n{chr(0x1f4cf)} PHASE 1: CALIBRATION")
    print(f"Running jobs one at a time until one succeeds to measure VRAM...")

    cal_result = None
    cal_peak_vram = 0
    cal_tokens = 0
    calibration_idx = -1

    skipped_in_cal = 0
    for cal_idx, cal_job in enumerate(jobs):
        cal_job_name = cal_job['name']

        # Skip already-completed jobs -- they give no VRAM data
        if config.get('skip_existing', False) and has_existing_output(cal_job_name):
            print(f"  \u23ed Skipping {cal_job_name} (already done)")
            completed_jobs.append({'name': cal_job_name, 'skipped': True, 'success': True,
                                   'structures': 0, 'time': 0, 'gpu_peak': 0,
                                   'tokens': count_job_tokens(cal_job, processor)})
            skipped_in_cal += 1
            continue

        cal_tokens = count_job_tokens(cal_job, processor)
        print(f"\n   Calibration attempt {cal_idx + 1}/{len(jobs)}: {cal_job['name']} ({cal_tokens} tokens)")

        cal_result = run_single_prediction_sequential(cal_job, cal_idx + 1, len(jobs))

        if cal_result['success']:
            completed_jobs.append(cal_result)
            calibration_idx = cal_idx
            cal_peak_vram = cal_result.get('gpu_peak', 0)
            print(f"\n   {chr(0x2705)} Calibration succeeded on job '{cal_job['name']}' -- peak VRAM: {cal_peak_vram:.2f} GB")
            break
        else:
            failed_jobs.append(cal_result)
            print(f"   {chr(0x26a0)}  Job '{cal_job['name']}' failed -- trying next job for calibration...")
            clear_gpu_cache()
            time.sleep(2)
            continue

    remaining_jobs = jobs[calibration_idx + 1:] if calibration_idx >= 0 else []

    if calibration_idx < 0:
        if skipped_in_cal == len(jobs):
            print(f"\n{chr(0x2705)} All {len(jobs)} jobs already completed. Nothing to run.")
        else:
            print(f"\n{chr(0x274c)} ALL {len(jobs)} jobs failed. No successful calibration possible.")
    elif not remaining_jobs:
        print("\nOnly one job - calibration complete, nothing more to run.")
    elif cal_peak_vram <= 0:
        # No VRAM data - fall back to sequential
        print(f"\nCould not measure VRAM during calibration. Falling back to sequential.")
        for job_idx, job in enumerate(remaining_jobs, calibration_idx + 2):
            result = run_single_prediction_sequential(job, job_idx, len(jobs))
            if result['success']:
                completed_jobs.append(result)
            else:
                failed_jobs.append(result)
            clear_gpu_cache()
            time.sleep(1)
    elif cal_peak_vram > total_gpu_vram * vram_limit:
        # Single job already uses most of the GPU - sequential only
        print(f"\nCalibration job used {cal_peak_vram:.1f} GB " +
              f"(>{total_gpu_vram * vram_limit:.1f} GB safe limit)")
        print(f"   Single job already fills GPU. Running remaining {len(remaining_jobs)} jobs sequentially.")

        for job_idx, job in enumerate(remaining_jobs, calibration_idx + 2):
            result = run_single_prediction_sequential(job, job_idx, len(jobs))
            if result['success']:
                completed_jobs.append(result)
            else:
                if result.get('can_retry'):
                    retried_jobs.append(job['name'])
                    clear_gpu_cache(); time.sleep(2)
                    result = run_single_prediction_sequential(job, job_idx, len(jobs), retry_with_safer=True)
                    if result['success']:
                        completed_jobs.append(result)
                    else:
                        failed_jobs.append(result)
                else:
                    failed_jobs.append(result)
            clear_gpu_cache()
            time.sleep(1)
    else:
        # --- PHASE 2: ESTIMATE REMAINING JOBS ---
        vram_per_token = cal_peak_vram / cal_tokens
        model_overhead = cal_peak_vram * 0.3  # minimum floor

        print(f"\nPHASE 2: VRAM ESTIMATION")
        print(f"   Calibration: {cal_peak_vram:.2f} GB peak for {cal_tokens} tokens")
        print(f"   Rate: {vram_per_token:.4f} GB/token")
        print(f"   Model overhead floor: {model_overhead:.2f} GB")
        print(f"   Available for parallel: {total_gpu_vram * vram_limit:.1f} GB")

        job_estimates = []
        for job in remaining_jobs:
            tokens = count_job_tokens(job, processor)
            estimated_vram = max(
                vram_per_token * tokens * 1.15,  # 15% safety margin
                model_overhead                    # minimum floor
            )
            job_estimates.append({
                'job': job,
                'tokens': tokens,
                'estimated_vram': estimated_vram
            })
            print(f"   {job['name']}: {tokens} tokens -> ~{estimated_vram:.1f} GB estimated")

        # --- PHASE 3: PARALLEL EXECUTION ---
        print(f"\nPHASE 3: PARALLEL EXECUTION")
        safe_limit = total_gpu_vram * vram_limit
        emergency = total_gpu_vram * vram_limit

        # Parse seeds list
        seeds_list = [s.strip() for s in config['seeds'].split(',')]

        # State tracking
        pending_estimates = list(enumerate(job_estimates, calibration_idx + 2))  # (job_number, estimate)
        running_procs = []  # list of dicts with process info
        emergency_triggered = False

        gpu_monitor.start()

        def try_launch_next():
            """Try to launch the next pending job if VRAM allows."""
            if not pending_estimates:
                return False

            current_vram = gpu_monitor.get_current()
            # Use HIGHER of actual GPU or estimated running (nvidia-smi lags behind)
            estimated_running = sum(p['estimated_vram'] for p in running_procs)
            effective_vram = max(current_vram, estimated_running)
            job_number, est = pending_estimates[0]
            needed = est['estimated_vram']

            if effective_vram + needed < safe_limit:
                pending_estimates.pop(0)
                job = est['job']
                job_name = job['name']

                # Skip existing check
                if config.get('skip_existing', False) and has_existing_output(job_name):
                    print(f"  \u23ed Skipping {job_name}: output already exists")
                    completed_jobs.append({'success': True, 'name': job_name, 'skipped': True,
                                          'structures': 0, 'time': 0, 'gpu_peak': 0,
                                          'tokens': count_job_tokens(job, processor)})
                    return True

                job_dir = get_unique_job_dir(job_name)
                os.makedirs(job_dir, exist_ok=True)


                json_content = processor._generate_json(job)

                # Inject pre-computed MSA paths (all-or-nothing)
                msas_injected = False
                msa_folder_val = msa_folder
                if msa_folder_val:
                    precomp_msas = find_precomputed_msas(job, msa_folder_val)
                    if precomp_msas:
                        import json as _json_msa
                        seq_to_name = {}
                        for seq in job.get('sequences', []):
                            if seq.get('type', '').lower() == 'protein' and seq.get('name'):
                                seq_to_name[seq['sequence']] = seq['name']
                        json_data = _json_msa.loads(json_content)
                        for item in json_data:
                            for seq_entry in item.get('sequences', []):
                                if 'proteinChain' not in seq_entry:
                                    continue
                                pc = seq_entry['proteinChain']
                                chain_name = seq_to_name.get(pc.get('sequence', ''))
                                if chain_name and chain_name in precomp_msas:
                                    msa_path = precomp_msas[chain_name]
                                    pc['pairedMsaPath'] = msa_path
                                    ind_dir = os.path.join(os.path.dirname(msa_folder_val), 'individual')
                                    ind_file = os.path.join(ind_dir, f"{chain_name}.a3m")
                                    pc['unpairedMsaPath'] = ind_file if os.path.isfile(ind_file) else msa_path
                        json_content = _json_msa.dumps(json_data, indent=2)
                        msas_injected = True
                json_file = os.path.join(job_dir, f"{job_name}.json")
                with open(json_file, 'w') as f:
                    f.write(json_content)

                cmd = build_protenix_cmd(json_file, job_dir, config, msas_injected=msas_injected)

                try:
                    proc = subprocess.Popen(
                        cmd, shell=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, universal_newlines=True
                    )
                except Exception as e:
                    print(f"Failed to launch {job_name}: {e}")
                    failed_jobs.append({
                        'success': False, 'name': job_name,
                        'error': f'Launch failed: {e}', 'time': 0, 'gpu_peak': 0
                    })
                    return try_launch_next()  # try next

                # Non-blocking reader
                out_q = queue.Queue()
                reader = threading.Thread(target=_reader_thread, args=(proc.stdout, out_q), daemon=True)
                reader.start()

                info = {
                    'process': proc, 'job': job, 'job_name': job_name,
                    'job_dir': job_dir, 'json_file': json_file,
                    'job_number': job_number, 'tokens': est['tokens'],
                    'estimated_vram': needed, 'start_time': time.time(),
                    'output_queue': out_q, 'error_messages': [],
                    'log_lines': [],
                    'msa_paths': precomp_msas if msas_injected else None,
                }
                running_procs.append(info)
                gpu_monitor.register_pid(proc.pid, job_name)

                print(f"\nLaunched job {job_number}/{len(jobs)}: {job_name} "
                      f"(est. {needed:.1f} GB, GPU: {effective_vram:.1f}/{safe_limit:.1f} GB)")

                time.sleep(2)  # brief pause for VRAM allocation
                return True
            return False

        # Launch initial jobs
        print(f"\nLaunching parallel jobs...")
        while try_launch_next():
            pass

        if running_procs:
            print(f"\n{len(running_procs)} job(s) running, {len(pending_estimates)} pending")
        else:
            print(f"\nCould not launch any parallel jobs")

        # Main monitoring loop
        while running_procs or pending_estimates:
            for info in list(running_procs):
                # Drain output queue
                while not info['output_queue'].empty():
                    try:
                        line = info['output_queue'].get_nowait()
                        if not line:
                            continue
                        info['log_lines'].append(line)
                        if 'error' in line.lower() and ('nan' in line.lower() or 'inf' in line.lower()):
                            info['error_messages'].append(line)
                        if parse_protenix_progress(line):
                            cur = gpu_monitor.get_current()
                            print(f"   [{info['job_name']}] {line[:80]} [GPU: {cur:.1f} GB]")
                        elif 'error' in line.lower() or 'warning' in line.lower():
                            print(f"   WARNING [{info['job_name']}] {line[:80]}")
                    except queue.Empty:
                        break

                # Check completion
                rc = info['process'].poll()
                if rc is not None:
                    running_procs.remove(info)
                    job_time = time.time() - info['start_time']
                    pid_peak = gpu_monitor.unregister_pid(info['process'].pid)

                    # Final drain: capture any lines buffered between last poll and exit
                    while not info['output_queue'].empty():
                        try:
                            line = info['output_queue'].get_nowait()
                            if line:
                                info['log_lines'].append(line)
                        except queue.Empty:
                            break

                    try:
                        if rc == 0:
                            print(f"\n{chr(0x2705)} Completed: {info['job_name']} ({job_time:.1f}s, VRAM: {pid_peak:.1f} GB)")
                            result = process_completed_job(
                                info['job_name'], info['job_dir'], info['json_file'],
                                job_time, pid_peak, drive, gdrive_folder_id, seeds_list,
                                log_lines=info['log_lines'],
                                msa_paths=info.get('msa_paths')
                            )
                            result['tokens'] = info['tokens']
                            if result['success']:
                                completed_jobs.append(result)
                            else:
                                failed_jobs.append(result)
                        else:
                            print(f"\n{chr(0x274c)} Failed: {info['job_name']} (exit code {rc}, {job_time:.1f}s, VRAM: {pid_peak:.1f} GB)")
                            error_files = read_error_files(info['job_dir'], info['job_name'])
                            if error_files:
                                for ef in error_files[:1]:
                                    print(f"   {ef['content'][:200]}")
                            failed_jobs.append({
                                'success': False, 'name': info['job_name'],
                                'error': f'Exit code {rc}', 'error_messages': info['error_messages'],
                                'time': job_time, 'gpu_peak': pid_peak
                            })
                    except Exception as e:
                        print(f"  ERROR processing {info['job_name']}: {e}")
                        failed_jobs.append({
                            'success': False, 'name': info['job_name'],
                            'error': str(e), 'time': job_time, 'gpu_peak': pid_peak
                        })

                    # Try launching next
                    clear_gpu_cache()
                    time.sleep(1)
                    gpu_monitor.reset_peak()

                    if not emergency_triggered:
                        while try_launch_next():
                            pass
                        if running_procs or pending_estimates:
                            print(f"   Status: {len(running_procs)} running, "
                                  f"{len(pending_estimates)} pending, "
                                  f"{len(completed_jobs)} done, {len(failed_jobs)} failed")

            # Emergency VRAM check
            current_vram = gpu_monitor.get_current()
            if current_vram > emergency and not emergency_triggered:
                emergency_triggered = True
                print(f"\nEMERGENCY: GPU VRAM at {current_vram:.1f}/{total_gpu_vram:.1f} GB")
                print(f"   Stopping new launches. Waiting for {len(running_procs)} running jobs...")

            time.sleep(2)

        peak_vram = gpu_monitor.stop()

        # If there are remaining jobs after emergency, run sequentially
        if pending_estimates:
            print(f"\nRunning {len(pending_estimates)} remaining jobs sequentially after emergency...")
            for job_number, est in pending_estimates:
                result = run_single_prediction_sequential(est['job'], job_number, len(jobs))
                if result['success']:
                    completed_jobs.append(result)
                else:
                    failed_jobs.append(result)
                clear_gpu_cache()
                time.sleep(1)


# ============================================================
# FINAL ipSAE BATCH ANALYSIS
# ============================================================
ipsae_batch_url = None
if (ipsae_available and config.get('enable_ipsae')
        and config.get('ipsae_batch_analysis') and len(completed_jobs) > 1):
    print("\n" + "=" * 60)
    print("FINAL ipSAE BATCH ANALYSIS")
    print("=" * 60)

    ipsae_batch_output = "/content/ipsae_batch_results"
    os.makedirs(ipsae_batch_output, exist_ok=True)

    cmd_parts = [
        "ipsae-batch", "/content",
        "--backend", "protenix",
        "--output_dir", ipsae_batch_output,
        "--pae_cutoff", str(config.get('ipsae_pae_cutoff', 10.0)),
        "--dist_cutoff", str(config.get('ipsae_dist_cutoff', 10.0)),
        "--cores", str(os.cpu_count() or 2),
    ]
    if config.get('ipsae_per_residue'):
        cmd_parts.append("--per_residue")
    if config.get('ipsae_per_contact'):
        cmd_parts.append("--per_contact")
    if config.get('ipsae_select_residues'):
        cmd_parts.extend(["--select-residues", config['ipsae_select_residues']])

    print(f"Analyzing {len(completed_jobs)} completed jobs...")
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=600)
        if result.returncode == 0:
            batch_files = [f for f in os.listdir(ipsae_batch_output) if not f.startswith('.')]
            print(f"Batch analysis complete: {len(batch_files)} files")
            for bf in sorted(batch_files):
                size = os.path.getsize(os.path.join(ipsae_batch_output, bf))
                print(f"   {bf} ({size//1024}KB)")

            # Upload to GDrive
            if drive and gdrive_folder_id:
                batch_zip = "/content/ipsae_batch_results.zip"
                with zipfile.ZipFile(batch_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
                    for root, dirs, files_list in os.walk(ipsae_batch_output):
                        for f in files_list:
                            fp = os.path.join(root, f)
                            arcname = os.path.join("ipsae_batch_results", os.path.relpath(fp, ipsae_batch_output))
                            zipf.write(fp, arcname)
                ipsae_batch_url = upload_to_gdrive(drive, batch_zip, gdrive_folder_id, "ipsae_batch")
                print(f"Uploaded: {ipsae_batch_url}")
            else:
                print(f"Saved locally: {ipsae_batch_output}/")
        else:
            print(f"Batch analysis failed (rc={result.returncode})")
            if result.stderr:
                print(f"   {result.stderr[:300]}")
    except subprocess.TimeoutExpired:
        print("Batch analysis timed out (600s)")
    except Exception as e:
        print(f"Batch analysis error: {e}")

# ============================================================
# FINAL SUMMARY
# ============================================================
total_time = time.time() - batch_start_time

print("\n" + "=" * 60)
print("BATCH PROCESSING COMPLETE")
print("=" * 60)
print(f"Ended: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total duration: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print(f"\nCompleted: {len(completed_jobs)}/{len(jobs)} jobs")
print(f"Failed: {len(failed_jobs)}/{len(jobs)} jobs")
if retried_jobs:
    print(f"Retried with safer settings: {len(retried_jobs)} jobs")

if completed_jobs:
    print(f"\nSuccessful jobs:")
    for job in completed_jobs:
        time_str = f"{job['time']:.1f}s" if job['time'] < 60 else f"{job['time']/60:.1f}m"
        gpu_str = f" | Peak GPU: {job['gpu_peak']:.2f} GB" if job.get('gpu_peak') else ""
        structs = job.get('structures', '?')
        print(f"  - {job['name']}: {structs} structures ({time_str}{gpu_str})")
        if job.get('url'):
            print(f"    {job['url']}")

if failed_jobs:
    print(f"\nFailed jobs:")
    for job in failed_jobs:
        error = job.get('error', 'Unknown error')[:80]
        time_str = f"{job['time']:.1f}s" if job.get('time', 0) < 60 else f"{job.get('time', 0)/60:.1f}m"
        print(f"  - {job['name']}: {error} ({time_str})")

# GPU statistics
if gpu_available and completed_jobs:
    gpu_peaks = [j['gpu_peak'] for j in completed_jobs if j.get('gpu_peak') and j['gpu_peak'] > 0]
    if gpu_peaks:
        print(f"\nGPU Peak Statistics:")
        print(f"   Average: {sum(gpu_peaks)/len(gpu_peaks):.2f} GB")
        print(f"   Highest: {max(gpu_peaks):.2f} GB")
        print(f"   Lowest: {min(gpu_peaks):.2f} GB")
        print(f"   Capacity: {total_gpu_vram:.1f} GB")
        print(f"   Peak utilization: {(max(gpu_peaks)/total_gpu_vram)*100:.1f}%")

# ipSAE Analysis Summary
if config.get('enable_ipsae'):
    ipsae_count = sum(1 for j in completed_jobs if j.get('ipsae_ran'))
    print(f"\nipSAE Interface Analysis:")
    print(f"   Per-job: {ipsae_count}/{len(completed_jobs)} jobs analyzed")
    if ipsae_batch_url:
        print(f"   Batch comparison: {ipsae_batch_url}")

if completed_jobs:
    avg_time = sum(j['time'] for j in completed_jobs) / len(completed_jobs)
    print(f"\nAverage time per job: {avg_time:.1f}s ({avg_time/60:.1f} min)")

if drive and gdrive_folder_id:
    print(f"\nAll results uploaded to: {gdrive_folder_name}")
    print(f"   https://drive.google.com/drive/folders/{gdrive_folder_id}")
else:
    print(f"\nAll results saved locally in job directories")

print("\nAll done!")

